In [ ]:
from repnr_ind import repetition_neurons, induction_heads, utils
import induction_heads
import visualize

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import random
from huggingface_hub import login
import json


repetition_neurons.seed_everything(41)
device = "cuda" if torch.cuda.is_available() else "cpu"
def load_data(path_to_data):
    tmp = [json.loads(line) for line in open(path_to_data)]
    return tmp
repetition_dataset = load_data('/workspace/nhi/data/natural/llama_31_8b/llama_31_8b.jsonl')[:1000]#workspace/nhi/data/natural/qwen25_7b/qwen25_7b.jsonl  / //workspace/nhi/data/natural/gemma2_9b/gemma2_9b.jsonlworkspace/nhi/data/natural/llama_2_13b/llama_2_13b.jsonl
model_name = 'meta-llama/Llama-3.1-8B'#'google/gemma-2-9b'#'Qwen/Qwen2.5-7B'#'meta-llama/Llama-2-13b-hf'#
model, tokenizer = repetition_neurons.load_model(model_name = model_name, seed=41)


In [ ]:
import importlib

importlib.reload(repetition_neurons)

sortedNeurons = repetition_neurons.find_sorted_neurons(model, tokenizer, base_data=repetition_dataset, seed=41)
#sortedHeads = induction_heads.compute_prefix_matching_scores(model, tokenizer)

In [ ]:
sortedNeurons[2]

# Visualization for repnr

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ——— ACL-style rcParams, with a built-in serif ———
plt.style.use('seaborn-v0_8')
plt.rcParams.update({
    ## switch to DejaVu Serif (ships with Matplotlib)
    #'font.family':      'serif',
    #'font.serif':       ['DejaVu Serif'],
    'font.size':        8,
    'axes.titlesize':   8,
    'axes.labelsize':   8,
    'xtick.labelsize':  7,
    'ytick.labelsize':  7,
    'legend.fontsize':  7,
    'figure.figsize':   (3.3, 2.2),
    'axes.linewidth':   0.5,
    'lines.linewidth':  1.0,
    'lines.markersize': 2,
    'legend.frameon':   False,
})

num_layers = model.config.num_hidden_layers
rel_pos    = np.linspace(0, 1, num_layers)

fig, ax = plt.subplots()
styles = [
  (250, 'o', '-', 'C0'),
  (500, 's', '-','C1'),
  (1000,'^','-','C2'),
  (1250,'d','-','C3')
]
for topk, marker, ls, col in styles:
    layers = [info['neuron'][0] for info in sortedNeurons[:topk]]
    counts = [layers.count(l) for l in range(num_layers)]
    ax.plot(
      rel_pos, counts,
      color=col, linestyle=ls, marker=marker,
      markevery=4,      # one marker every 4 layers
      alpha=0.75,       # slightly transparent
      label=str(topk)
    )

ax.set_xlabel("Relative Layer Position")
ax.set_ylabel("Number of Neurons")
ticks = np.arange(0.0, 1.01, 0.2)
ax.set_xticks(ticks)
ax.set_xticklabels([f"{t:.1f}" for t in ticks])
ax.yaxis.grid(True, linestyle='--', linewidth=0.4, alpha=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(title="Top K", loc='upper left', ncol=2)
plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ——— ACL-style with seaborn base but only horizontal grids ———
#plt.style.use('seaborn-v0_8')
plt.style.use('default')
plt.rcParams.update({
    # typography & sizing
    'font.size':        8,
    'axes.titlesize':   8,
    'axes.labelsize':   8,
    'xtick.labelsize':  7,
    'ytick.labelsize':  7,
    'legend.fontsize':  7,
    'figure.figsize':   (3.3, 2.2),
    # lines & markers
    'axes.linewidth':   0.5,
    'lines.linewidth':  1.0,
    'lines.markersize': 2,
    'legend.frameon':   False,
    # make sure default grid is on (we’ll refine it below)
    'axes.grid':        True,
    'grid.linestyle':   '--',
    'grid.linewidth':   0.4,
    'grid.alpha':       0.6,
})

# data setup
num_layers = model.config.num_hidden_layers
rel_pos    = np.linspace(0, 1, num_layers)

fig, ax = plt.subplots()

# plot each Top-K curve with spaced markers
styles = [
  (250, 'o', '-', 'C0'),
  (500, 's', '-','C1'),
  (1000,'^','-','C2'),
  (1250,'d','-','C3')
]
for topk, marker, ls, col in styles:
    layers = [info['neuron'][0] for info in sortedNeurons[:topk]]
    counts = [layers.count(l) for l in range(num_layers)]
    ax.plot(
        rel_pos, counts,
        color=col, linestyle=ls, marker=marker,
        markevery=4,           # one marker every 4 layers
        alpha=0.75,
        label=f"Top {topk}"
    )

# refine axes
ax.set_xlabel("Relative Layer Position")
ax.set_ylabel("Number of Neurons")

# ticks at 0.0, 0.2, …, 1.0
ticks = np.arange(0.0, 1.01, 0.2)
ax.set_xticks(ticks)
ax.set_xticklabels([f"{t:.1f}" for t in ticks])

# turn off vertical grid, keep only horizontal
ax.xaxis.grid(False)
ax.yaxis.grid(True)

# remove top & right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# legend & layout
ax.legend(loc='upper left', ncol=2, title="Top K neurons")
plt.tight_layout()
#plt.savefig("/workspace/nhi/figure/natural/llama_3.1_8b/neuron_distribution_acl_1.png",
#            format="png",
#            dpi=300,
#            bbox_inches="tight")
plt.show()


# Continue code

In [ ]:
sortedHeads2 = induction_heads.get_avg_score(model, tokenizer, data=repetition_dataset)

In [ ]:
importlib.reload(induction_heads)
sortedHeads = induction_heads.compute_prefix_matching_scores(model, tokenizer)

### plot head map

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch

def plot_prefix_matching_scores(avg_scores, vmin=None, vmax=None, save_path=None):
    """
    Plots a heatmap of prefix matching scores across layers and heads.
    
    Args:
        avg_scores (List[Tensor]): One tensor per layer of shape (num_heads,)
        vmin (float, optional): Min value for heatmap color scale.
        vmax (float, optional): Max value for heatmap color scale.
        save_path (str, optional): Path to save the figure. If None, just shows it.
    """
    # Stack into 2D matrix: (num_layers, num_heads)
    score_matrix = torch.stack(avg_scores).numpy()

    #plt.figure(figsize=(16, 6))
    ax = sns.heatmap(score_matrix, annot=False, vmin=vmin, vmax=vmax,
                xticklabels=True, yticklabels=True)

    plt.xlabel("Head")
    plt.ylabel("Layer")
    plt.title("Prefix Matching Score Heatmap")
    ax.invert_yaxis()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path)
        print(f"Saved to {save_path}")
    else:
        plt.show()

plot_prefix_matching_scores(sortedHeads)

In [ ]:
sortedHeads2_cpu = [t.cpu() for t in sortedHeads2]
plot_prefix_matching_scores(sortedHeads2_cpu)

## normal

In [ ]:
import utils

In [ ]:
import importlib

importlib.reload(repetition_neurons)
ab_dataset = load_data('/workspace/nhi/data/repetition_icl/2_apr/rep_10shot.jsonl')
ground_truths = [item['ground_truth'] for item in ab_dataset]


# Joint ablation

Ablate both repetition neurons and induction heads

In [ ]:
import importlib

importlib.reload(repetition_neurons)
importlib.reload(utils)
importlib.reload(induction_heads)

In [ ]:

rec_dual_results = utils.run_dual_ablation_study(
    model=model,
    tokenizer=tokenizer,
    dataset=ab_dataset,          
    sortedNeurons=sortedNeurons,
    id_avg_scores=sortedHeads,   
    percent_list=[1, 3, 5, 7, 10],
    segment_ranges=[(0, 0.2), (0.4, 0.6), (0.8, 1)],
    neurons_to_ablate_list=[20, 50, 100, 180, 250]
)

In [ ]:
rec_dual_results['top']['induction'][20][1][0].keys()

In [ ]:
import pandas as pd
from utils import extract_answer

def full_mode_extract(ab_dataset, ab_results, nor_, shots=5):
    ground_truths = [item['ground_truth'] for item in ab_dataset]
    queries = [item['query'] for item in ab_dataset]

    # Base DataFrame
    df_base = pd.DataFrame({
        'query': queries,
        'ground_truth': ground_truths,
        'nor': nor_
    })

    # List to store new columns
    new_columns = []

    for nr_mode, nr_mode_results in ab_results.items():
        for id_mod, hd_mode_results in nr_mode_results.items():
            for nr_num, nr_num_results in hd_mode_results.items():
                for id_segment, id_segment_results in nr_num_results.items():
                    for nr_segment_results in id_segment_results:
                        segment_range = nr_segment_results["segment"]
                        generated_texts = nr_segment_results["results"]
                        key = f"{segment_range.replace('->', '_')}"
                        cl_name = f"ab_{nr_mode}_{id_mod}_{nr_num}_{id_segment}_{key}"

                        extracted_list = [
                            extract_answer(text, shots) for text in generated_texts
                        ]

                        new_columns.append(pd.DataFrame({cl_name: extracted_list}))

    # Concatenate all at once to avoid fragmentation
    df_full = pd.concat([df_base] + new_columns, axis=1)

    return df_full


d1 = full_mode_extract(ab_dataset, rec_dual_results, nor_, shots=10)

In [ ]:
foo_df = d1[d1['ground_truth'] == 'Foo']
ab_top_induction_cols = [col for col in foo_df.columns if col.startswith("ab_top_induction")]
ab_top_induction_cols = [col for col in ab_top_induction_cols if int(col.split("_")[4]) <= 1]

# Create a subset of the DataFrame with only these columns
ab_top_induction_df = foo_df[['query', 'ground_truth', 'nor'] + ab_top_induction_cols]

In [ ]:
for col in ab_top_induction_cols:
    ab_top_induction_df[f"{col}_correct"] = ab_top_induction_df[col].apply(
        lambda x: 'Foo' in x  # True if 'Foo' is in the cell, False otherwise
    )

In [ ]:
# Calculate accuracy for each column
accuracy_summary_foo = {}
for col in ab_top_induction_cols:
    accuracy_summary_foo[col] = ab_top_induction_df[f"{col}_correct"].mean()

# Convert to a DataFrame for easier analysis
accuracy_summary_foo_df = pd.DataFrame.from_dict(accuracy_summary_foo, orient='index', columns=['accuracy'])
accuracy_summary_foo_df.index.name = "condition"
accuracy_summary_foo_df.reset_index(inplace=True)

In [ ]:
accuracy_summary_foo_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extract parameters from column names
accuracy_summary_foo_df['nr_num'] = accuracy_summary_foo_df['condition'].apply(
    lambda x: int(x.split("_")[3])  # Extract the number of neurons ablated
)
accuracy_summary_foo_df['segment_range'] = accuracy_summary_foo_df['condition'].apply(
    lambda x: x[-5:]  # Extract the segment range
)

# Create line plot
plt.figure(figsize=(12, 6))
sns.lineplot(data=accuracy_summary_foo_df, x="nr_num", y="accuracy", hue="segment_range", markers=True)
plt.title("Accuracy vs. Number of Neurons Ablated (Foo Prediction)")
plt.xlabel("Number of Neurons Ablated")
plt.ylabel("Accuracy")
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_accuracy_per_class_with_segments(d1, save_path=None):
    labels = ["Foo", "Bar"]
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

    plt.style.use('seaborn-v0_8')
    fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

    handles = []
    title_map = {'Foo': 'Pattern Query', 'Bar': 'Non-pattern Query'}

    for idx, label in enumerate(labels):
        df = d1[d1['ground_truth'] == label]

        # Detect ablation columns for both modes
        ab_induction_cols = [col for col in df.columns if col.startswith("ab_top_induction")]
        ab_random_cols = [col for col in df.columns if col.startswith("ab_top_random")]

        # Filter columns with a specific percentage (e.g., b == 3)
        ab_induction_cols = [col for col in ab_induction_cols if int(col.split("_")[4]) == 3]
        ab_random_cols = [col for col in ab_random_cols if int(col.split("_")[4]) == 3]

        # Combine both modes into one DataFrame
        ab_cols = ab_induction_cols + ab_random_cols
        ab_df = df[['query', 'ground_truth', 'nor'] + ab_cols].copy()

        # Compute accuracy for each column
        for col in ab_cols:
            ab_df[f"{col}_correct"] = ab_df[col].apply(lambda x: label in x)

        accuracy = {col: ab_df[f"{col}_correct"].mean() for col in ab_cols}
        acc_df = pd.DataFrame.from_dict(accuracy, orient='index', columns=['accuracy']).reset_index()
        acc_df.rename(columns={'index': 'condition'}, inplace=True)

        # Extract ablation number, segment, and mode
        acc_df['nr_num'] = acc_df['condition'].apply(lambda x: int(x.split("_")[3]))
        acc_df['segment_range'] = acc_df['condition'].apply(lambda x: "_".join(x.split("_")[-2:]))
        acc_df['mode'] = acc_df['condition'].apply(lambda x: "induction" if "induction" in x else "random")

        segments = acc_df['segment_range'].unique()
        ax = axes[idx]

        for i, seg in enumerate(segments):
            # Filter data for the current segment
            df_seg = acc_df[acc_df['segment_range'] == seg]

            # Plot induction mode (solid line)
            df_induction = df_seg[df_seg['mode'] == 'induction']
            if not df_induction.empty:
                df_induction = df_induction.groupby('nr_num', as_index=False)[['accuracy']].mean().sort_values('nr_num')
                line, = ax.plot(
                    df_induction['nr_num'],
                    df_induction['accuracy'],
                    marker='o',
                    linestyle='-',
                    color=colors[i % len(colors)],
                    label=f"induction - segment {seg.replace('_', '→')}",
                    linewidth=2,
                    markersize=8,
                )
                if idx == 0:
                    handles.append(line)

            # Plot random mode (dashed line)
            df_random = df_seg[df_seg['mode'] == 'random']
            #print(df_random)
            if not df_random.empty:
                df_random = df_random.groupby('nr_num', as_index=False)[['accuracy']].mean().sort_values('nr_num')
                line, = ax.plot(
                    df_random['nr_num'],
                    df_random['accuracy'],
                    marker='x',
                    linestyle='--',
                    color=colors[i % len(colors)],
                    label=f"random - segment {seg.replace('_', '→')}",
                    linewidth=2,
                    markersize=8,
                )
                if idx == 0:
                    handles.append(line)

        ax.set_title(f"{title_map[label]}", fontsize=16, fontweight='bold')
        ax.set_xlabel("Number of Neurons Ablated", fontsize=14)
        ax.set_ylabel("Accuracy", fontsize=14)
        ax.set_xticks(sorted(acc_df['nr_num'].unique()))
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.3, linestyle='--')

    # Shared legend
    legend = fig.legend(
        handles=handles,
        title="Induction Head Ablation Modes and Segment Ranges",
        loc='lower center',
        bbox_to_anchor=(0.5, -0.15),
        ncol=min(len(handles), 8),
        fontsize=12,
        title_fontsize=14,
        frameon=True,
        edgecolor='black'
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if save_path is not None:
        plt.savefig(f"{save_path}/foo_bar_segments.png", bbox_inches='tight')
    plt.show()


# Call the function
plot_accuracy_per_class_with_segments(d1)

In [ ]:
d1.columns

In [ ]:
classes = ['Foo', 'Bar']
overview = {}

for cls in classes:
    df_filtered = d1[d1['ground_truth'] == cls]
    total_rows = len(df_filtered)
    correct_counts = df_filtered.apply(lambda col: col.astype(str).str.contains(f": {cls}", na=False).sum())
    accuracy = correct_counts / total_rows
    overview[cls] = {
            "total_rows": total_rows,
            "accuracy": accuracy.to_dict()
        }

In [ ]:
foo_acc = overview["Foo"]["accuracy"]

# Extract nor_ accuracy from d1
nor_acc = (d1['ground_truth'] == "Foo") & (d1['nor'].astype(str).str.contains(": Foo", na=False))
nor_accuracy = nor_acc.sum() / (d1['ground_truth'] == "Foo").sum()

# Filter features with lower accuracy than nor_
lower_than_nor = {k: v for k, v in foo_acc.items() if v < nor_accuracy}

# Show results
import pprint
pprint.pprint(lower_than_nor)

In [ ]:
nor_answer = utils.generate_answer(model, tokenizer, ab_dataset)
nor_ =  utils.extract_answers_nor(nor_answer, shots = 10)

In [ ]:
ab_hd_result = induction_heads.ablate_attention_heads(model, tokenizer, sortedHeads, ab_dataset)
hd = utils.extract_answers_ab(ab_hd_result, shots=10)

In [ ]:
ab_hd_result2 = induction_heads.ablate_attention_heads(model, tokenizer, sortedHeads2, ab_dataset)
hd2 = utils.extract_answers_ab(ab_hd_result2, shots=10)

In [ ]:
print(type(ab_nr_all_result))

In [ ]:
d1.to_csv("/workspace/nhi/data/ab_result/llama31_8b/10_shot_idnr/rep_idnr_extracted.csv", index=False)

In [ ]:
from collections import defaultdict
extracted = defaultdict(lambda: defaultdict(list))
    
for entry in ab_nr_all_result:
    for ablation_type, percent_dict in entry.items():
        for percent, outputs in percent_dict.items():
            extracted[ablation_type][percent].extend(outputs)

In [ ]:
ab_nr_layer_results = load_data("/workspace/nhi/data/ab_result/llama31_8b/10_shot/rep_ab_nr_layer_result.jsonl")
ab_nr_all_result = load_data("/workspace/nhi/data/ab_result/llama31_8b/10_shot/rep_ab_nr_all_result.jsonl")
ab_hd = load_data("/workspace/nhi/data/ab_result/llama31_8b/10_shot/rep_ab_hd_result.jsonl")
#nr_all = utils.extract_answers_ab(ab_nr_all_result, shots = 10)
df_1 = utils.analyze_exp_1(ab_dataset, nor_, nr_all, hd)
overview_results = utils.analyze_class_overview(df_1)
overview_results_top, overview_results_random = utils.split_results_by_mode(overview_results)

In [ ]:
nr = load_data("/workspace/nhi/data/ab_result/llama31_8b/10_shot/rep_nor_answer.jsonl")
nr  = utils.extract_answers_nor(nr, shots = 10)

In [ ]:
ab_hd = load_data("/workspace/nhi/data/ab_result/llama31_8b/10_shot/rep_ab_hd_result.jsonl")
hd = utils.extract_answers_ab(ab_hd_result, shots=10)

In [ ]:
df_1 = utils.analyze_exp_1(ab_dataset, nr, nr_all, hd)
overview_results1 = utils.analyze_class_overview(df_1)
overview_results_top1, overview_results_random1 = utils.split_results_by_mode(overview_results1)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_accuracy_from_overview(overview_results, save_path=None):
    # Define labels and colors
    labels = ["Foo", "Bar"]
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    line_styles = {'induction': '-', 'random': '--'}

    # Set plotting style
    plt.style.use('seaborn-v0_8')
    fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

    handles = []
    title_map = {'Foo': 'Pattern Query', 'Bar': 'Non-pattern Query'}

    for idx, label in enumerate(labels):
        # Extract accuracy data for the current label
        accuracy_data = overview_results[label]['accuracy']

        # Separate induction and random head data
        induction_data = {k: v for k, v in accuracy_data.items() if k.startswith("ab_id_top")}
        random_data = {k: v for k, v in accuracy_data.items() if k.startswith("ab_id_random")}

        # Extract number of neurons ablated (nr_num) from keys
        nr_nums_induction = sorted([int(k.split("_")[3]) for k in induction_data.keys()])
        nr_nums_random = sorted([int(k.split("_")[3]) for k in random_data.keys()])

        ax = axes[idx]

        # Plot induction data (solid line)
        induction_acc = [induction_data[f"ab_id_top_{nr}"] for nr in nr_nums_induction]
        line, = ax.plot(
            nr_nums_induction,
            induction_acc,
            marker='o',
            linestyle=line_styles['induction'],
            color=colors[0],
            label="Induction Heads",
            linewidth=2,
            markersize=8,
        )
        if idx == 0:
            handles.append(line)

        # Plot random data (dashed line)
        random_acc = [random_data[f"ab_id_random_{nr}"] for nr in nr_nums_random]
        line, = ax.plot(
            nr_nums_random,
            random_acc,
            marker='x',
            linestyle=line_styles['random'],
            color=colors[1],
            label="Random Heads",
            linewidth=2,
            markersize=8,
        )
        if idx == 0:
            handles.append(line)

        # Add normal accuracy as a horizontal line
        normal_accuracy = accuracy_data['nor']
        ax.axhline(
            y=normal_accuracy,
            color='gray',
            linestyle='--',
            linewidth=1.5,
            label="Normal Accuracy"
        )

        # Configure subplot
        ax.set_title(f"{title_map[label]}", fontsize=16, fontweight='bold')
        ax.set_xlabel("Number of Neurons Ablated", fontsize=14)
        ax.set_ylabel("Accuracy", fontsize=14)
        ax.set_xticks(sorted(set(nr_nums_induction + nr_nums_random)))
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.3, linestyle='--')

    # Shared legend
    legend = fig.legend(
        handles=handles,
        title="Modes",
        loc='lower center',
        bbox_to_anchor=(0.5, -0.15),
        ncol=len(handles),
        fontsize=12,
        title_fontsize=14,
        frameon=True,
        edgecolor='black'
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if save_path is not None:
        plt.savefig(f"{save_path}/foo_bar_accuracy.png", bbox_inches='tight')
    plt.show()


# Call the function
plot_accuracy_from_overview(overview_results1)

In [ ]:
overview_results1

In [ ]:
overview_results_random

In [ ]:
utils.save_to_jsonl(rec_dual_results, "/workspace/nhi/data/ab_result/llama31_8b/10_shot_idnr/rep_dual.jsonl")

# additional measure average activation value of repetition neurons when ablating induction head exp

# sst2 task

## layer-wise

In [ ]:
from tqdm.notebook import tqdm
import importlib
import utils
importlib.reload(repetition_neurons)
importlib.reload(utils)

def layer_wise(model, tokenizer, sortedNeurons, shot=5, K_values=[20, 50, 100, 180, 250], seed=42):
    tasks = ['sst2']
    d = {}
    for task in tqdm(tasks):
        file_data = f"/workspace/nhi/data/sst2/icl/{task}_{shot}shot_sul.jsonl"
        ab_dataset = load_data(file_data)
        ground_truths = [item['ground_truth'] for item in ab_dataset]
        nor_answer = utils.generate_answer(model, tokenizer, ab_dataset, seed)
        ab_nr_layer_result = repetition_neurons.run_segment_ablation_study_2phase(model, tokenizer, 
                                                                                   ab_dataset, sortedNeurons, 
                                                                                   neurons_to_ablate_list=K_values,seed=seed)
        nor_ =  utils.extract_answers_nor_sst2(nor_answer)
        nr_layer = utils.extract_answers_ab_layer(ab_nr_layer_result)
        d[task] = {
            'ground_truths': ground_truths,
            'nr_layer':      nr_layer,
            'nor_':          nor_,
        }
    return d

d = layer_wise(model, tokenizer, sortedNeurons, shot=5, K_values=[20, 50, 100, 180, 250, 350,500],seed=42)

In [ ]:
import json

output_path = "/workspace/nhi/data/ab_result/llama31_8b/extended/sst2/neuron_5shot_sul_42.jsonl"
with open(output_path, "w", encoding="utf-8") as fout:
    for task, res in d.items():
        # embed the task name in the record
        record = {"task": task, **res}
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
def compute_ablation_accuracies(results_dict, mode='top', K=250, sul=False):
    def class_accuracy(gt_list, pred_list, cls):
        idxs = [i for i, gt in enumerate(gt_list) if gt == cls]
        if not idxs:
            return float('nan')
        correct = sum(1 for i in idxs if pred_list[i] == cls)
        return correct / len(idxs)

    accuracies = {}
    for task, res in results_dict.items():
        gts = res['ground_truths']
        
        # Get unique classes present in ground truths
        unique_classes = set(gts)
        
        # get segment-wise predictions at the given mode and K
        try:
            seg_preds = res['nr_layer'][mode][K]
        except KeyError:
            raise KeyError(f"No data for task={task}, mode={mode}, K={K}")

        task_acc = {}

        # Compute accuracy per segment
        for segment, preds in seg_preds.items():
            seg_acc = {cls: class_accuracy(gts, preds, cls) for cls in unique_classes}
            task_acc[segment] = seg_acc

        # Compute accuracy for 'nor_' predictions
        nor_acc = {cls: class_accuracy(gts, res['nor_'], cls) for cls in unique_classes}
        for cls, acc in nor_acc.items():
            task_acc[f'{cls}_nor'] = acc

        accuracies[task] = task_acc

    return accuracies

In [ ]:
ab_dataset[0]['prompt'].split('\n')[-1].split(": ")[-1]

In [ ]:
all_task_result = compute_ablation_accuracies(d)

In [ ]:
all_task_result

In [ ]:
def compute_ablation_recall(
    d,
    mode,
    K,
    tasks=None,
    segments=None,
):
    if tasks is None:
        tasks = list(d.keys())
    if segments is None:
        segments = ['0_0.2', '0.4_0.6', '0.8_1.0']
    
    # Get all accuracies using the updated function
    all_accs = compute_ablation_accuracies(d, mode, K)
    
    # Extract recall values dynamically
    recall = {}
    for task in tasks:
        task_recall = {}
        # Get class names from one of the segments or nor entries
        # (assuming all have the same classes per task)
        sample_seg = next(iter(all_accs[task]))  # pick any segment
        classes = list(all_accs[task][sample_seg].keys())

        for seg in segments:
            seg_recall = {cls: all_accs[task][seg][cls] for cls in classes}
            task_recall[seg] = seg_recall

        recall[task] = task_recall

    return recall

In [ ]:
K_values=[100, 250, 500]
modes = ['top','random']
recalls_by_modeK = {
    m: {K: compute_ablation_recall(d, m, K) for K in K_values}
    for m in modes
}
baseline_recall = recalls_by_modeK['top'][K_values[0]]

In [ ]:
baseline_recall

In [ ]:
recalls_by_modeK

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_all_tasks_ablation_both_modes(
    d,
    K_values=[20, 50, 100, 180, 250],
    tasks=None,
    marker_size=2
):
    if tasks is None:
        tasks = list(d.keys())
    n_tasks = len(tasks)

    modes = ['top', 'random']
    segments = ['0_0.2', '0.4_0.6', '0.8_1.0']
    markers = {'0_0.2': 'o', '0.4_0.6': 's', '0.8_1.0': 'x'}
    linestyles = {'top': '-', 'random': '--'}
    colors = {'0_0.2': 'C0', '0.4_0.6': 'C1', '0.8_1.0': 'C2'}
    alpha_map = {'top': 1.0, 'random': 0.5}

    xs = [0] + K_values

    # Precompute accuracies
    accs_by_modeK = {
        m: {K: compute_ablation_accuracies(d, m, K) for K in K_values}
        for m in modes
    }
    baseline_acc = accs_by_modeK['top'][K_values[0]]

    sample_task = tasks[0]
    sample_seg = next(iter(segments))

    # Access the first segment dict and get its keys → these are actual class names
    classes = list(baseline_acc[sample_task][sample_seg].keys())
    num_classes = len(classes)

    # Set up figure
    plt.style.use('default')
    plt.rcParams.update({
        'font.size': 9,
        'axes.titlesize': 9,
        'axes.labelsize': 12,
        'xtick.labelsize': 7,
        'ytick.labelsize': 7,
        'legend.fontsize': 7,
        'figure.figsize': (n_tasks * 4, num_classes * 2),
        'lines.markersize': 4,
        'legend.frameon': True,
        'axes.grid': False,
    })

    fig, axes = plt.subplots(
        num_classes, n_tasks,
        sharey='row',
        figsize=(n_tasks * 4, num_classes * 2),
        squeeze=False,
        constrained_layout=True
    )
    handles, labels = [], []

    for col, task in enumerate(tasks):
        # Get baseline values per class
        base_accs = {
            cls: baseline_acc[task][f"{cls}_nor"]
            for cls in classes
        }

        # Build curves
        curves = {
            cls: {
                m: {
                    seg: [base_accs[cls]] for seg in segments
                } for m in modes
            } for cls in classes
        }

        for m in modes:
            for K in K_values:
                accs = accs_by_modeK[m][K][task]
                for seg in segments:
                    for cls in classes:
                        curves[cls][m][seg].append(accs[seg][cls])

        # Plot each class in its own row
        for row, cls in enumerate(classes):
            ax = axes[row, col]
            for m in modes:
                for seg in segments:
                    line, = ax.plot(
                        xs,
                        curves[cls][m][seg],
                        linestyle=linestyles[m],
                        marker=markers[seg],
                        color=colors[seg],
                        markersize=marker_size,
                        alpha=alpha_map[m]
                    )
                    # Collect handles and labels only once
                    if col == 0:
                        low, high = seg.split('_')
                        interval = f"[{float(low):.1f}, {float(high):.1f}]"
                        label = f"{m} {interval}"
                        if label not in labels:  # Avoid duplicates
                            handles.append(line)
                            labels.append(label)
            ax.set_xticks(xs)
            if col == 0:
                ax.set_ylabel(f'{cls} Recall', fontsize=9)
            ax.set_title(task, fontsize=9)

    # Shared legend
    fig.legend(
        handles, labels,
        title="Neuron Type - Segment",
        ncol=len(modes),
        fontsize=8,
        title_fontsize=8,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.2)
    )

    fig.supxlabel("Number of Deactivated Neurons", fontsize=9)
    plt.savefig("/workspace/nhi/figure/extended/sst2/llama31_8b/41_neuron_5shot_sul.png",
                format="png", dpi=300, bbox_inches="tight")
    plt.show()
plot_all_tasks_ablation_both_modes(d, K_values=[50, 100, 180, 250, 350,500])#, mode='top', K_values=[20,50,100,180,250])[50, 150, 250, 350, 500]

In [ ]:
def compute_ablation_precision(results_dict, mode='top', K=250):
    def class_precision(gt_list, pred_list, cls):
        # Get indices where the model predicted the class `cls`
        idxs = [i for i, pred in enumerate(pred_list) if pred == cls]
        if not idxs:
            return float('nan')
        # Count how many of those predictions are correct (true positives)
        correct = sum(1 for i in idxs if gt_list[i] == cls)
        return correct / len(idxs)

    precisions = {}
    for task, res in results_dict.items():
        gts = res['ground_truths']
        
        # Extract unique classes from ground truths
        unique_classes = set(gts)
        
        try:
            seg_preds = res['nr_layer'][mode][K]
        except KeyError:
            raise KeyError(f"No data for task={task}, mode={mode}, K={K}")

        task_prec = {}

        # Compute precision for each segment
        for segment, preds in seg_preds.items():
            seg_prec = {cls: class_precision(gts, preds, cls) for cls in unique_classes}
            task_prec[segment] = seg_prec

        # Compute precision for 'nor_' baseline
        nor_prec = {cls: class_precision(gts, res['nor_'], cls) for cls in unique_classes}
        for cls, prec in nor_prec.items():
            task_prec[f'{cls}_nor'] = prec

        precisions[task] = task_prec

    return precisions

In [ ]:
def plot_all_tasks_ablation_both_modes_precision(
    d,
    K_values=[20, 50, 100, 180, 250],
    tasks=None,
    marker_size=2
):
    if tasks is None:
        tasks = list(d.keys())
    n_tasks = len(tasks)

    modes = ['top', 'random']
    segments = ['0_0.2', '0.4_0.6', '0.8_1.0']
    markers = {'0_0.2': 'o', '0.4_0.6': 's', '0.8_1.0': 'x'}
    linestyles = {'top': '-', 'random': '--'}
    colors = {'0_0.2': 'C0', '0.4_0.6': 'C1', '0.8_1.0': 'C2'}
    alpha_map = {'top': 1.0, 'random': 0.5}

    xs = [0] + K_values

    # Precompute precision values
    prec_by_modeK = {
        m: {K: compute_ablation_precision(d, m, K) for K in K_values}
        for m in modes
    }
    baseline_prec = prec_by_modeK['top'][K_values[0]]

    # Get unique classes dynamically
    sample_task = tasks[0]
    sample_seg = next(iter(segments))
    classes = list(baseline_prec[sample_task][sample_seg].keys())
    num_classes = len(classes)

    # Setup figure
    plt.style.use('default')
    plt.rcParams.update({
        'font.size': 9,
        'axes.titlesize': 9,
        'axes.labelsize': 12,
        'xtick.labelsize': 7,
        'ytick.labelsize': 7,
        'legend.fontsize': 7,
        'figure.figsize': (n_tasks * 4, num_classes * 2),
        'lines.markersize': 4,
        'legend.frameon': True,
    })

    fig, axes = plt.subplots(
        num_classes, n_tasks,
        sharey='row',
        figsize=(n_tasks * 4, num_classes * 2),
        squeeze=False,
        constrained_layout=True
    )

    handles, labels = [], []

    for col, task in enumerate(tasks):
        # Get baseline values for each class
        base_precs = {
            cls: baseline_prec[task][f"{cls}_nor"]
            for cls in classes
        }

        # Build curves
        curves = {
            cls: {
                m: {
                    seg: [base_precs[cls]] for seg in segments
                } for m in modes
            } for cls in classes
        }

        for m in modes:
            for K in K_values:
                precs = prec_by_modeK[m][K][task]
                for seg in segments:
                    for cls in classes:
                        curves[cls][m][seg].append(precs[seg][cls])

        # Plot each class in its own row
        for row, cls in enumerate(classes):
            ax = axes[row, col]

            for m in modes:
                for seg in segments:
                    line, = ax.plot(
                        xs,
                        curves[cls][m][seg],
                        linestyle=linestyles[m],
                        marker=markers[seg],
                        color=colors[seg],
                        markersize=marker_size,
                        alpha=alpha_map[m]
                    )
                    if col == 0:
                        low, high = seg.split('_')
                        interval = f"[{float(low):.1f}, {float(high):.1f}]"
                        label = f"{m} {interval}"
                        if label not in labels:
                            handles.append(line)
                            labels.append(label)

            ax.set_xticks(xs)
            if col == 0:
                ax.set_ylabel(f'{cls} Precision', fontsize=9)
            ax.set_title(task, fontsize=9)

    # Shared legend
    fig.legend(
        handles, labels,
        title="Neuron Type - Segment",
        ncol=len(modes),
        fontsize=8,
        title_fontsize=8,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.2)
    )

    fig.supxlabel("Number of Deactivated Neurons", fontsize=9)
    plt.savefig("/workspace/nhi/figure/extended/sst2/llama31_8b/41_neuron_5shot_sul_precision.png",
                format="png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
plot_all_tasks_ablation_both_modes_precision(d, K_values=[50, 100, 180, 250, 350, 500])

## ab head

In [ ]:
segment_ranges = [(0, 0.2), (0.4, 0.6), (0.8, 1.0)]
neurons_by_layer_position = {}
total_layers = len(model.model.layers)
tmp_nr = {}
for neuron_info in sortedNeurons:
        layer_idx, neuron_idx = neuron_info['neuron']
        relative_position = layer_idx / total_layers  
        if relative_position not in neurons_by_layer_position:
            neurons_by_layer_position[relative_position] = []
        neurons_by_layer_position[relative_position].append(neuron_info)
for i, (start, end) in enumerate(segment_ranges):#tqdm(enumerate(segment_ranges), desc="Processing segment ranges", unit="segment"):
    #print(f"\nProcessing segment {i+1}: {start} -> {end}...")
    segment_neurons = []
    segment_layer_indices = set()  
    for layer_position, neurons in neurons_by_layer_position.items():
        if start <= layer_position < end:
            segment_neurons.extend(neurons)
            segment_layer_indices.add(int(layer_position * len(model.model.layers)))
    segment_neurons_sorted = sorted(segment_neurons, key=lambda x: x['diffs'], reverse=True)
    top_neurons_to_ablate_list = [neuron['neuron'] for neuron in segment_neurons_sorted[:250]]
    import random
    random.shuffle(segment_neurons)
    random_neurons_to_ablate_list = [neuron['neuron'] for neuron in segment_neurons[:250]]
    #if (start, end) not in tmp_nr:
    #    tmp_nr[(start, end)] = []
    tmp_nr[(start, end)]={'top': top_neurons_to_ablate_list, 'ran': random_neurons_to_ablate_list}

In [ ]:
for segment, segment_nrs in tmp_nr.items():
    #print(segment_nrs.keys())
    for mode, segment_nr in segment_nrs.items():
        print(segment_nr)

In [ ]:
if tokenizer.bos_token_id is None:
    # Any valid integer works; using EOS keeps things consistent
    tokenizer.add_special_tokens({"bos_token": tokenizer.eos_token})
    model.resize_token_embeddings(len(tokenizer))

In [ ]:
from tqdm import tqdm
import importlib
import utils
importlib.reload(induction_heads)
importlib.reload(utils)
importlib.reload(repetition_neurons)



def id_ab(model, tokenizer, tmp_nr,shot=10, seed=42):
    tasks = ['rec']#'ceb','wsq3','wsq4']# 'rep', 'rec',
    d = {}
    sortedHeads = induction_heads.compute_prefix_matching_scores(model, tokenizer)
    for task in tqdm(tasks):
        file_data = f"/workspace/nhi/data/repetition_icl/2_apr/{task}_{shot}shot.jsonl"#f"/workspace/nhi/data/sst2/icl/{task}_{shot}shot_sul.jsonl"
        ab_dataset = load_data(file_data)
        ground_truths = [item['ground_truth'] for item in ab_dataset]
        print("\n\n\n TASK: ",task)
        for segment, segment_nrs in tmp_nr.items():
            for mode, segment_nr in segment_nrs.items():
                nor_answer, nor_prob = utils.generate_answer(model, tokenizer, ab_dataset, seed, rep_neuron=segment_nr)
                ab_hd_result, rep_acts = induction_heads.ablate_attention_heads_8(model, tokenizer, sortedHeads, 
                                                                                percent_list=[3], test_dataset=ab_dataset, 
                                                                                seed=seed, rep_neuron=segment_nr)
                #induction_heads.ablate_attention_heads_loglikelihood(model, tokenizer, sortedHeads, ab_dataset)
                nor_ =  utils.extract_answers_nor(nor_answer)
                hd = utils.extract_answers_ab(ab_hd_result)
                d[task] = {
                    'ground_truths': ground_truths,
                    'hd': hd,
                    'nor_': nor_,
                    #"ab_hd_result": ab_hd_result[""]
                }
                print(f"{segment}_{mode}: \n Normal probe: {round(nor_prob,4)} \n Rp result: {rep_acts}")
    return d, nor_prob, rep_acts
hd_result = id_ab(model, tokenizer, tmp_nr, shot=10, seed = 41)
#hd_result = id_ab(model, tokenizer, shot=10, seed = 42)

In [ ]:
hd_result[1]

In [ ]:
hd_result[2]

In [ ]:
hd_result[1]['31:14080']

In [ ]:
hd_result[2]['random'][3]['31:14080']

In [ ]:
import json

output_path = "/workspace/nhi/data/ab_result/llama31_8b/extended/sst2/head_5shot_sul_42.jsonl"
with open(output_path, "w", encoding="utf-8") as fout:
    for task, res in hd_result.items():
        # embed the task name in the record
        record = {"task": task, **res}
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
def compute_head_ablation_accuracy(hd_result):
    accuracies = {}
    for task, res in hd_result.items():
        gts     = res['ground_truths']
        nor_p   = res['nor_']
        n       = len(gts)
        # baseline (no ablation) accuracy
        nor_acc = sum(1 for gt, p in zip(gts, nor_p) if gt == p) / n

        hd_acc = {}
        for mode, counts_map in res['hd'].items():
            mode_acc = {}
            for k, preds in counts_map.items():
                # pair up ground truth and preds
                length = min(n, len(preds))
                preds = [_.strip() for _ in preds]
                correct = sum(1 for i in range(length) if gts[i] == preds[i])
                mode_acc[k] = correct / length if length > 0 else float('nan')
            hd_acc[mode] = mode_acc

        accuracies[task] = {
            'nor_acc': nor_acc,
            'hd_acc':  hd_acc
        }

    return accuracies

# Example usage:
#hd_result_extract = id_ab(model, tokenizer)
head_accs = compute_head_ablation_accuracy(hd_result)

In [ ]:
head_accs

In [ ]:
head_accs

In [ ]:
def compute_head_ablation_accuracy_by_class(hd_result, classes=None):
    out = {}
    for task, res in hd_result.items():
        gts   = res['ground_truths']
        nor_p = res['nor_']

        # determine classes
        task_classes = classes if classes is not None else sorted(set(gts))

        # build index lists per class
        class_idxs = {
            cls: [i for i, gt in enumerate(gts) if gt == cls]
            for cls in task_classes
        }

        # baseline per-class accuracy
        nor = {}
        for cls, idxs in class_idxs.items():
            if idxs:
                nor[cls] = sum(1 for i in idxs if nor_p[i] == cls) / len(idxs)
            else:
                nor[cls] = float('nan')

        # head-drop accuracies
        hd_acc = {}
        for mode, cmap in res['hd'].items():
            mode_map = {}
            for k, preds in cmap.items():
                m = {}
                for cls, idxs in class_idxs.items():
                    # only include positions where gt==cls and i < len(preds)
                    valid_idxs = [i for i in idxs if i < len(preds)]
                    if valid_idxs:
                        m[cls] = sum(1 for i in valid_idxs if preds[i] == cls) / len(valid_idxs)
                    else:
                        m[cls] = float('nan')
                mode_map[k] = m
            hd_acc[mode] = mode_map

        out[task] = {
            'nor':    nor,
            'hd_acc': hd_acc
        }

    return out

ab_hd_acc = compute_head_ablation_accuracy_by_class(hd_result)

In [ ]:
ab_hd_acc

In [ ]:
import matplotlib.pyplot as plt

def plot_head_ablation_results(accs):
    tasks    = list(accs.keys())
    k_values = sorted(accs[tasks[0]]['hd_acc']['induction'].keys())
    xs       = [0] + k_values   # prepend 0 for the baseline

    # create a 1×N grid, force it not to squeeze
    fig, axes = plt.subplots(1, len(tasks),
                             sharey=True,
                             figsize=(len(tasks)*2, 3),
                             squeeze=False)

    # flatten to a simple 1-D list of Axes
    axes = axes.flatten()

    for ax, task in zip(axes, tasks):
        data = accs[task]
        nor  = data['nor_acc']
        ind  = [data['hd_acc']['induction'][k] for k in k_values]
        rnd  = [data['hd_acc']['random'][k]    for k in k_values]

        # prepend baseline
        ind_plot = [nor] + ind
        rnd_plot = [nor] + rnd

        ax.set_box_aspect(1)
        ax.plot(xs, ind_plot, 'o-',  label='Induction', markersize=4)
        ax.plot(xs, rnd_plot, 's--', label='Random',    markersize=4)

        ax.set_title(task, fontsize=9)
        ax.set_xticks(xs)
        ax.set_xlabel("Percentage of Heads Ablated", fontsize=8)
        ax.grid(axis='y', linestyle='--', alpha=0.5)

    # first subplot gets the shared y-label
    axes[0].set_ylabel("Recall", fontsize=9)

    # shared legend beneath all plots
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc='lower center',
        ncol=2,
        frameon=True,
        fontsize=8,
        title="Ablation Type",
        title_fontsize=9,
        bbox_to_anchor=(0.5, -0.08)
    )

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.savefig("/workspace/nhi/figure/extended/sst2/llama31_8b/41_head_5shot_sul.png",format="png",
            dpi=300,
            bbox_inches="tight")
    plt.show()
plot_head_ablation_results(head_accs)

In [ ]:
def plot_head_ablation_by_class(accs, classes=None):
    tasks = list(accs.keys())
    # infer classes if not provided
    if classes is None:
        classes = sorted(accs[tasks[0]]['nor'].keys())

    # get the sorted head-indices
    k_vals = sorted(accs[tasks[0]]['hd_acc']['induction'].keys())
    xs     = [0] + k_vals

    n_rows = len(classes)
    n_cols = len(tasks)
    fig, axes = plt.subplots(
        n_rows, n_cols,
        sharey=True,
        figsize=(n_cols * 2, n_rows * 2),
        squeeze=False
    )

    for col, task in enumerate(tasks):
        data = accs[task]
        nor  = data['nor']

        for row, cls in enumerate(classes):
            ax = axes[row, col]
            # build the two curves
            ind_curve = [nor[cls]] + [data['hd_acc']['induction'][k][cls] for k in k_vals]
            rnd_curve = [nor[cls]] + [data['hd_acc']['random'][k][cls]    for k in k_vals]

            # plot with explicit labels
            ax.plot(xs, ind_curve, marker='o', linestyle='-', label="Induction")
            ax.plot(xs, rnd_curve, marker='s', linestyle='--', label="Random")

            if col == 0:
                ax.set_ylabel(f"{cls} Recall", fontsize=8)
            if row == 0:
                ax.set_title(task, fontsize=8)

            ax.set_xticks(xs)
            ax.grid(axis='y', linestyle='--', alpha=0.5)

    # pull handles & labels from any one of the subplots
    handles, labels = axes[0,0].get_legend_handles_labels()

    fig.legend(
        handles, labels,
        ncol=2,
        loc='upper center',
        bbox_to_anchor=(0.5, 0.1),
        frameon=True, fontsize=8,
        title="Ablation Type", title_fontsize=9
    )

    fig.supxlabel("Number of Heads Ablated", fontsize=9, y=0.12)
    plt.tight_layout(rect=[0, 0.07, 1, 1])
    plt.savefig(
        "/workspace/nhi/figure/extended/sst2/llama31_8b/41_head_class_5shot_sul.png",
        format="png", dpi=300, bbox_inches="tight"
    )
    plt.show()


plot_head_ablation_by_class(ab_hd_acc)

## joint ab

In [ ]:
from tqdm import tqdm
import importlib
import utils
importlib.reload(repetition_neurons)
importlib.reload(utils)

def joint_ab(model, tokenizer, sortedNeurons, shot=10, seed=42):
    tasks = ['sst2']
    d = {}
    sortedHeads = induction_heads.compute_prefix_matching_scores(model, tokenizer)
    for task in tqdm(tasks):
        file_data = f"/workspace/nhi/data/sst2/icl/{task}_{shot}shot_sul.jsonl"
        ab_dataset = load_data(file_data)
        ground_truths = [item['ground_truth'] for item in ab_dataset]
        nor_answer = utils.generate_answer(model, tokenizer, ab_dataset, seed)
        
        dual_results = utils.run_dual_ablation_study(
            model=model,
            tokenizer=tokenizer,
            dataset=ab_dataset,          
            sortedNeurons=sortedNeurons,
            id_avg_scores=sortedHeads,   
            percent_list=[1, 3, 5],
            segment_ranges=[(0, 0.2), (0.4, 0.6), (0.8, 1.0)],
            neurons_to_ablate_list=[100, 250],
            shot=shot,
        )
        nor_ =  utils.extract_answers_nor(nor_answer)
        #ab_dual = utils.extract_answers_ab_layer(ab_dual_result, shots=shot)
        d[task] = {
            'ground_truths': ground_truths,
            'ab_dual':      dual_results,
            'nor_':          nor_,
        }
    return d

joint_rs = joint_ab(model, tokenizer, sortedNeurons, shot=10, seed=41)


In [ ]:
joint_rs.keys()

In [ ]:
import json

output_path = "/workspace/nhi/data/ab_result/llama31_8b/extended/sst2/joint_5shot_sul_41.jsonl"
with open(output_path, "w", encoding="utf-8") as fout:
    for task, res in joint_rs.items():
        # embed the task name in the record
        record = {"task": task, **res}
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
def compute_dual_ablation_accuracies_by_class(joint_results, classes=None):
    """
    From joint_results (with keys 'ground_truths', 'nor_', 'ab_dual'),
    compute for each task:
      - baseline Foo / Bar accuracy
      - for each rep_mode ∈ {'top','random'}, each mask_mode ∈ {'induction','random'},
        each k ∈ neurons_to_ablate_list, each pct ∈ percent_list:
          segment → { 'Foo': acc, 'Negative': acc }
    """
    out = {}
    for task, info in joint_results.items():
        gts = info['ground_truths']
        n = len(gts)

        # determine classes
        task_classes = classes if classes is not None else sorted(set(gts))

        # build indices for each class
        class_idxs = {
            cls: [i for i, gt in enumerate(gts) if gt == cls]
            for cls in task_classes
        }

        # baseline accuracies per class
        nor_preds = info['nor_']
        baseline = {}
        for cls, idxs in class_idxs.items():
            if idxs:
                baseline[cls] = sum(1 for i in idxs if nor_preds[i] == cls) / len(idxs)
            else:
                baseline[cls] = float('nan')

        # ablations
        ablation = {}
        for rep_mode, rep_block in info['ab_dual'].items():
            ablation[rep_mode] = {}
            for mask_mode, mask_block in rep_block.items():
                ablation[rep_mode][mask_mode] = {}
                for k, pct_block in mask_block.items():
                    ablation[rep_mode][mask_mode][k] = {}
                    for pct, seg_list in pct_block.items():
                        seg_map = {}
                        for seg_res in seg_list:
                            seg = seg_res['segment']
                            preds = seg_res['results']
                            # compute per-class acc for this segment
                            cls_acc = {}
                            for cls, idxs in class_idxs.items():
                                if idxs:
                                    cls_acc[cls] = sum(1 for i in idxs if preds[i] == cls) / len(idxs)
                                else:
                                    cls_acc[cls] = float('nan')
                            seg_map[seg] = cls_acc
                        ablation[rep_mode][mask_mode][k][pct] = seg_map

        out[task] = {
            'baseline': baseline,
            'ablation': ablation
        }

    return out

# Example usage:
dual_accs_class = compute_dual_ablation_accuracies_by_class(joint_rs)
# Now dual_accs['rep']['baseline'] → {'Foo':…, 'Negative':…}
# and dual_accs['rep']['ablation']['top']['induction'][100][1]['0->0.2'] → {'Foo':…, 'Negative':…}


In [ ]:
dual_accs_class

In [ ]:
def extract_last_segment_delta(dual_accs,
                               rep_mode='top',
                               mask_mode='induction',
                               neuron_count=250,
                               pct=3,
                               segment='0.8->1',
                               ndigits=3):
    out = {}
    for task, info in dual_accs.items():
        base = info['baseline']            # {'Foo':…, 'Bar':…}
        after = ( info['ablation']
                      [rep_mode]
                      [mask_mode]
                      [neuron_count]
                      [pct]
                      [segment] )
        out[task] = (
            round(after['Positive'] - base['Positive'], ndigits),#'Foo': 
            round(after['Negative'] - base['Negative'], ndigits) #'Bar': 
        )
    return out


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

tasks = ['sst2']
top_deltas = extract_last_segment_delta(dual_accs_class,
                                    rep_mode='top',
                                    mask_mode='induction',
                                    neuron_count=250,
                                    pct=3,
                                    segment='0.8->1.0')
random_deltas = extract_last_segment_delta(dual_accs_class,
                                    rep_mode='random',
                                    mask_mode='induction',
                                    neuron_count=250,
                                    pct=3,
                                    segment='0.8->1.0')
# unpack
pattern_top  = [top_deltas[t][0] for t in tasks]
pattern_rand = [random_deltas[t][0] for t in tasks]
non_top      = [top_deltas[t][1] for t in tasks]
non_rand     = [random_deltas[t][1] for t in tasks]

# styling for one‐column width (≈3.3in)
plt.rcParams.update({
    'figure.figsize': (3.3, 3.0),
    'axes.labelsize': 8,
    'xtick.labelsize': 9,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'lines.linewidth': 0.9,
})

x = np.arange(len(tasks))
width = 0.35

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

# Pattern‐accuracy Δ
ax1.bar(x - width/2, pattern_top,  width, label='Top',    alpha=0.8)
ax1.bar(x + width/2, pattern_rand, width, label='Random', alpha=0.8)
ax1.set_ylabel('Δ Positive')
#ax1.legend(ncol=2, loc='upper right', frameon=False)

# Non‐pattern Δ
ax2.bar(x - width/2, non_top,  width, label='Repetition',    alpha=0.8)
ax2.bar(x + width/2, non_rand, width, label='Random', alpha=0.8)
ax2.set_ylabel('Δ Negative')
ax2.set_xticks(x)
ax2.set_xticklabels(tasks, fontsize=10, rotation=0)
ax2.legend(ncol=2, loc='upper right', frameon=True, fontsize=6,
    framealpha=0.5,  title="Ablation Neuron", title_fontsize=6)

fig.supxlabel('Task', fontsize=9)
plt.tight_layout(pad=0.5)
# plt.savefig("/workspace/nhi/figure/natural/llama_3.1_8b/10_shot/43/dual_250_3_rndhd.png",format="png",
#             dpi=300,
#             bbox_inches="tight")

plt.show()


In [ ]:
from tqdm import tqdm
import importlib
import utils
import induction_heads
importlib.reload(repetition_neurons)
importlib.reload(utils)
importlib.reload(induction_heads)

In [ ]:
sortedNeurons[:250]

In [ ]:
# 1) Lấy mask cho induction heads (ví dụ p = 3%)
id_scores = induction_heads.compute_prefix_matching_scores(model, tokenizer)
num_layers = len(id_scores)
num_heads  = id_scores[0].shape[0]
masks = induction_heads.build_head_masks(id_scores, num_layers, num_heads, percent_list=[3], random_seed=42)
head_mask = masks["induction"][3]           # Tensor[num_layers, num_heads]  (0.0 = tắt)

# 2) Chọn top-K repetition neurons bạn đã có sẵn
K = 250
target_neurons = [x["neuron"] for x in sortedNeurons[:K]]

# 3) Dataset trùng với join ablation (SST-2 10-shot)
ab_dataset = load_data("/workspace/nhi/data/repetition_icl/28_mar/rep_10shot.jsonl")

# 4) Đo Δ activation khi ablate head
deltas, base_means, off_means = repetition_neurons.compute_rep_neuron_deltas(
    model, tokenizer, ab_dataset, target_neurons, head_mask=head_mask
)

# 5) (tuỳ chọn) Gộp theo segment để báo cáo
seg_delta = repetition_neurons.aggregate_by_segments(deltas, num_layers, segments=[(0.0,0.2),(0.4,0.6),(0.8,1.0)])
print("Δa mean theo segment:", seg_delta)


# Single element (neuron or head only)

## Layer-wise experiment

Layer-wise neurons ablation for abstract task

In [ ]:
from tqdm.notebook import tqdm
import importlib
import utils
importlib.reload(repetition_neurons)
importlib.reload(utils)

def layer_wise(model, tokenizer, sortedNeurons, shot=10, K_values=[20, 50, 100, 180, 250], seed=43):
    tasks = ['rep', 'rec', 'ceb', 'wsq1','wsq2']
    d = {}
    for task in tqdm(tasks):
        file_data = f"/workspace/nhi/repnr_ind/data/abstract_icl/{task}_{shot}shot.jsonl"
        ab_dataset = load_data(file_data)
        ground_truths = [item['ground_truth'] for item in ab_dataset]
        nor_answer = utils.generate_answer(model, tokenizer, ab_dataset, seed)
        ab_nr_layer_result = repetition_neurons.run_segment_ablation_study_2phase(model, tokenizer, 
                                                                                  ab_dataset, sortedNeurons, 
                                                                                  neurons_to_ablate_list=K_values,seed=seed)
        nor_ =  utils.extract_answers_nor(nor_answer, shots = shot)
        nr_layer = utils.extract_answers_ab_layer(ab_nr_layer_result, shots=shot)
        d[task] = {
            'ground_truths': ground_truths,
            'nr_layer':      nr_layer,
            'nor_':          nor_,
        }
    return d

d = layer_wise(model, tokenizer, sortedNeurons, shot=10, K_values=[20, 50, 100, 180, 250, 350,500],seed=41)

In [ ]:
import json

output_path = "/workspace/nhi/data/ab_result/llama31_8b/10_shot/layer_wise/layer_wise_results_41.jsonl"
with open(output_path, "w", encoding="utf-8") as fout:
    for task, res in d.items():
        # embed the task name in the record
        record = {"task": task, **res}
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
def compute_ablation_accuracies(results_dict, mode='top', K=250):
    def class_accuracy(gt_list, pred_list, cls):
        idxs = [i for i, gt in enumerate(gt_list) if gt == cls]
        if not idxs:
            return float('nan')
        correct = sum(1 for i in idxs if pred_list[i] == cls)
        return correct / len(idxs)

    accuracies = {}
    for task, res in results_dict.items():
        gts = res['ground_truths']
        # get segment‐wise predictions at the given mode and K
        try:
            seg_preds = res['nr_layer'][mode][K]
        except KeyError:
            raise KeyError(f"No data for task={task}, mode={mode}, K={K}")

        task_acc = {}
        for segment, preds in seg_preds.items():
            foo_acc = class_accuracy(gts, preds, 'Foo')
            bar_acc = class_accuracy(gts, preds, 'Bar')
            task_acc[segment] = {'Foo': foo_acc, 'Bar': bar_acc}
        task_acc['Foo_nor'] = class_accuracy(gts, res['nor_'], 'Foo')
        task_acc['Bar_nor'] = class_accuracy(gts, res['nor_'], 'Bar')

        accuracies[task] = task_acc

    return accuracies

print()

In [ ]:
all_task_result = compute_ablation_accuracies(d)

In [ ]:

def load_back(path="/workspace/nhi/data/ab_result/llama31_8b/10_shot/layer_wise/layer_wise_results.jsonl", exp='nr_layer'):
    d = []
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            if line.strip():                      # skip empty lines
                d.append(json.loads(line))

    d = {
        r["task"]: {k: v for k, v in r.items() if k != "task"}
        for r in d          
    }

    for task, task_results in d.items():
        for mode, mode_results in task_results[exp].items():
            if exp == "hd":
                for rank in ("induction", "random"):
                    mode_dict = task_results[exp][rank]
                    task_results[exp][rank] = {int(k): v               
                                                    for k, v in mode_dict.items()}
            else:
                for rank in ("top", "random"): 
                    if exp =='ab_dual':
                        for hd in ("induction", "random"):
                            mode_dict = task_results[exp][rank][hd]
                            task_results[exp][rank][hd] = {int(k): v               
                                                            for k, v in mode_dict.items()}
                            for p_head in task_results[exp][rank][hd].keys():
                                mode_dict = task_results[exp][rank][hd][p_head]
                                task_results[exp][rank][hd][p_head] ={int(k): v               
                                                            for k, v in mode_dict.items()}
                    else:
                        mode_dict = task_results[exp][rank]
                        task_results[exp][rank] = {int(k): v               
                                                        for k, v in mode_dict.items()}
    return d

#d = load_back() joint_rs['rep']['ab_dual']['top']['induction'].keys()

In [ ]:
d = load_back("/workspace/nhi/data/ab_result/llama31_8b/10_shot/layer_wise/layer_wise_results_42.jsonl", exp="nr_layer")

In [ ]:
all_task_result

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_ablation_accuracies(acc_dict, mode='top', K=250):

    tasks = list(acc_dict.keys())
    n_tasks = len(tasks)

    # the three ablated‐segment names
    segs = ['0_0.2', '0.4_0.6', '0.8_1.0']

    # build Foo (pattern) data: three segments + baseline Foo_nor
    foo_data = []
    for seg in segs:
        foo_data.append([acc_dict[t][seg]['Foo'] for t in tasks])
    foo_data.append([acc_dict[t]['Foo_nor'] for t in tasks])
    foo_data = np.array(foo_data)  # shape (4, n_tasks)

    # build Bar (non-pattern) data: three segments + baseline Bar_nor
    bar_data = []
    for seg in segs:
        bar_data.append([acc_dict[t][seg]['Bar'] for t in tasks])
    bar_data.append([acc_dict[t]['Bar_nor'] for t in tasks])
    bar_data = np.array(bar_data)

    labels = segs + ['Baseline']
    n_bars = len(labels)
    x = np.arange(n_tasks)
    width = 0.8 / n_bars

    fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True, figsize=(6.5, 3))

    # Plot Foo
    for i, label in enumerate(labels):
        ax1.bar(
            x + (i - (n_bars-1)/2) * width,
            foo_data[i],
            width,
            label=label
        )
    ax1.set_xticks(x)
    ax1.set_xticklabels(tasks)
    ax1.set_title(f"Pattern (Foo), mode={mode}, K={K}")
    ax1.set_ylabel("Accuracy")
    ax1.set_ylim(0, 1.05)
    ax1.legend(title="Segment / Baseline", fontsize=7, title_fontsize=8)

    # Plot Bar
    for i, label in enumerate(labels):
        ax2.bar(
            x + (i - (n_bars-1)/2) * width,
            bar_data[i],
            width,
            label=label
        )
    ax2.set_xticks(x)
    ax2.set_xticklabels(tasks)
    ax2.set_title(f"Non-pattern (Bar), mode={mode}, K={K}")
    ax2.set_ylim(0, 1.05)
    # no legend on second plot

    plt.tight_layout()
    plt.show()
plot_ablation_accuracies(all_task_result, mode='top', K=250)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_all_tasks_ablation(d, mode='top', K_values=[50, 150, 250, 350, 500], tasks=None):
    """
    d:               output of layer_wise(...)
    mode:            'top' or 'random'
    K_values:        list of ablation sizes
    tasks:           list of task names (defaults to d.keys())
    """
    # 1) Setup
    if tasks is None:
        tasks = list(d.keys())
    n_tasks = len(tasks)
    segments = ['0_0.2', '0.4_0.6', '0.8_1.0']
    styles   = ['o-', 's-', 'd-']  # one style per segment
    xs       = [0] + K_values       # x-axis tick positions
    
    # 2) Precompute accuracies for each K  
    accs_by_K = {K: compute_ablation_accuracies(d, mode, K) for K in K_values}
    # baseline “no ablation” comes out of any K’s result
    baseline_acc = accs_by_K[K_values[0]]
    plt.rcParams.update({
                'xtick.labelsize': 7,
                'ytick.labelsize': 7,
            })
    
    # 3) Make subplots: 2 rows × n_tasks columns
    fig, axes = plt.subplots(2, n_tasks, sharey=True, figsize=(n_tasks*2, 4))
    
    for col, task in enumerate(tasks):
        # baseline values for this task
        foo0 = baseline_acc[task]['Foo_nor']
        bar0 = baseline_acc[task]['Bar_nor']
        
        # gather per-segment curves
        pattern_curves    = {seg: [foo0] for seg in segments}
        nonpattern_curves = {seg: [bar0] for seg in segments}
        for K in K_values:
            accs = accs_by_K[K][task]
            for seg in segments:
                pattern_curves[seg].append(accs[seg]['Foo'])
                nonpattern_curves[seg].append(accs[seg]['Bar'])
        
        

        # Top row: Pattern (Foo)
        axp = axes[0, col]
        for seg, style in zip(segments, styles):
            axp.plot(xs, pattern_curves[seg], style, label=seg)
        if col == 0:
            axp.set_ylabel("Pattern Accuracy")
        axp.set_title(task)
        axp.set_xticks(xs)
        if col == n_tasks-1:
            axp.legend(title="Segment", fontsize=6, title_fontsize=7, loc="upper left")
        
        # Bottom row: Non-pattern (Bar)
        axn = axes[1, col]
        for seg, style in zip(segments, styles):
            axn.plot(xs, nonpattern_curves[seg], style, label=seg)
        if col == 0:
            axn.set_ylabel("Non-pattern Accuracy")
        axn.set_xticks(xs)
        if col == n_tasks-1:
            axn.legend(title="Segment", fontsize=6, title_fontsize=7, loc="upper left")
    
    # 4) Global axis labels
    fig.supxlabel("Number of Repetition Neurons Deactivated")
    plt.tight_layout()
    plt.show()
rename_map = {
    'wsq3': 'wsq1',
    'wsq4': 'wsq2',
}

# 2) build a new dict with keys replaced
d_renamed = {
    rename_map.get(task, task): results
    for task, results in d.items()
}

# 3) plot using the renamed dict
plot_all_tasks_ablation(d_renamed, mode='top', K_values=[50, 150, 250, 350, 500])

In [ ]:
import json
def load_data(path_to_data):
    tmp = [json.loads(line) for line in open(path_to_data)]
    return tmp
d = load_data("/workspace/nhi/data/ab_result/llama31_8b/10_shot/layer_wise/layer_wise_results.jsonl")

In [ ]:
d[0].keys()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_all_tasks_ablation_both_modes(
    d,
    K_values=[20,50,100,180,250],
    tasks=None,
    marker_size=2
):
    if tasks is None:
        tasks = list(d.keys())
    n_tasks  = len(tasks)
    
    modes      = ['top', 'random']
    segments   = ['0_0.2', '0.4_0.6', '0.8_1.0']
    markers    = {'0_0.2':'o', '0.4_0.6':'s', '0.8_1.0':'x'}
    linestyles = {'top':'-', 'random':'--'}
    colors     = {'0_0.2':'C0', '0.4_0.6':'C1', '0.8_1.0':'C2'}
    alpha_map  = {'top':1.0, 'random':0.5}
    
    xs = [0] + K_values
    
    # precompute
    accs_by_modeK = {
        m: {K: compute_ablation_accuracies(d, m, K) for K in K_values}
        for m in modes
    }
    baseline_acc = accs_by_modeK['top'][K_values[0]]
    
    #plt.rcParams.update({'xtick.labelsize':7, 'ytick.labelsize':7})
    plt.style.use('default')
    plt.rcParams.update({
        # typography & sizing
        'font.size':        9,
        'axes.titlesize':   9,
        'axes.labelsize':   12,
        'xtick.labelsize':  7,
        'ytick.labelsize':  7,
        'legend.fontsize':  7,
        'figure.figsize':   (3.3, 2.2),
        # lines & markers
        #'axes.linewidth':   0.5,
        #'lines.linewidth':  1.0,
        'lines.markersize': 4,
        'legend.frameon':   True,
        # make sure default grid is on (we’ll refine it below)
        'axes.grid':        False,
        #'grid.linestyle':   '--',
        #'grid.linewidth':   0.4,
        #'grid.alpha':       0.6,
    })
    fig, axes = plt.subplots(2, n_tasks, sharey=True, figsize=(n_tasks*2, 4))
    
    # collect one handle+label per mode‐segment for the shared legend
    handles, labels = [], []
    
    for col, task in enumerate(tasks):
        foo0 = baseline_acc[task]['Foo_nor']
        bar0 = baseline_acc[task]['Bar_nor']
        
        # build curves
        pattern_curves    = {m:{seg:[foo0] for seg in segments} for m in modes}
        nonpattern_curves = {m:{seg:[bar0] for seg in segments} for m in modes}
        for m in modes:
            for K in K_values:
                accs = accs_by_modeK[m][K][task]
                for seg in segments:
                    pattern_curves[m][seg].append(accs[seg]['Foo'])
                    nonpattern_curves[m][seg].append(accs[seg]['Bar'])
        
        # Row 0: Pattern (Foo)
        axp = axes[0, col]
        mode_names = {'top': 'repetition', 'random': 'random'}
        for m in modes:
            for seg in segments:
                line, = axp.plot(
                    xs,
                    pattern_curves[m][seg],
                    linestyle=linestyles[m],
                    marker=markers[seg],
                    color=colors[seg],
                    markersize=marker_size,
                    alpha=alpha_map[m]
                )
                # only register one handle/label per mode-segment
                if col == 0: 
                    low, high = seg.split('_')
                    interval   = f"[{float(low):.1f}, {float(high):.1f}]"
                    labels.append(f"{mode_names[m]} {interval}")
                    handles.append(line)
                    #labels.append(f"{mode_names[m]}-{seg}")
        if col == 0:
            axp.set_ylabel("Pattern Accuracy", fontsize=9)
        axp.set_title(task, fontsize=9)
        axp.set_xticks(xs)
        
        # Row 1: Non-pattern (Bar)
        axn = axes[1, col]
        for m in modes:
            for seg in segments:
                axn.plot(
                    xs,
                    nonpattern_curves[m][seg],
                    linestyle=linestyles[m],
                    marker=markers[seg],
                    color=colors[seg],
                    markersize=marker_size,
                    alpha=alpha_map[m]
                )
        if col == 0:
            axn.set_ylabel("Non-pattern Accuracy", fontsize=9)
        axn.set_xticks(xs)
    
    # shared legend centered below the plots
    fig.legend(
        handles, labels,
        title="neuron type-segment",
        ncol= len(segments)*len(modes),
        fontsize=8,
        title_fontsize=8,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.06)
    )
    
    # global X label
    fig.supxlabel("Number of Deactivated Neurons", fontsize=9, y = 0.05)
    plt.tight_layout()
    plt.savefig("/workspace/nhi/figure/natural/llama_3.1_8b/10_shot/43/ab_nrlayer_500nr.png",format="png",
            dpi=300,
            bbox_inches="tight")
    plt.show()


rename_map = {
    'wsq3': 'wsq1',
    'wsq4': 'wsq2',
}

# 2) build a new dict with keys replaced
d_renamed = {
    rename_map.get(task, task): results
    for task, results in d.items()
}

# 3) plot using the renamed dict
plot_all_tasks_ablation_both_modes(d_renamed, [50, 150, 250, 350, 500])#, mode='top', K_values=[20,50,100,180,250])

## Ablate induction heads

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_top3_distribution(avg_scores):
    """
    Plots the distribution of top 3% highest prefix matching scores (induction heads)
    across layers.
    
    Args:
        avg_scores (List[Tensor]): One tensor per layer of shape (num_heads,)
    """
    # Convert torch Tensors to numpy arrays
    layer_arrays = [layer.cpu().numpy() for layer in avg_scores]
    
    # Flatten all scores to compute the 97th percentile threshold
    all_scores = np.concatenate(layer_arrays)
    threshold = np.percentile(all_scores, 97)
    
    # Count heads above threshold per layer
    counts = [np.sum(layer > threshold) for layer in layer_arrays]
    print(np.sum(counts))
    layers = np.arange(len(avg_scores))
    num_layers = len(avg_scores)
    ticks = np.arange(0, num_layers, 2)
    
    
    # Plot bar chart
    plt.figure()
    plt.bar(layers, counts)
    plt.xlabel("Layer Index")
    plt.ylabel("Number of Induction Heads (Top 3%)")
    #plt.title("Distribution of Top 3% Prefix Matching Scores Across Layers")
    #plt.xticks(layers)
    plt.xticks(ticks)
    plt.show()


In [ ]:
import torch
import math
import numpy as np
import matplotlib.pyplot as plt

def plot_repetition_bar_induction_line(avg_scores, sorted_neurons, p=3, tick_step=2):
    """
    Bars: fraction of TOP-n_rep repetition neurons in each layer
    Line: fraction of TOP-p% induction heads *globally flattened*

    Args:
        avg_scores     (List[Tensor]): per‐layer prefix‐matching scores, shape (num_heads,)
        sorted_neurons (List[dict]):  output of find_sorted_neurons(), sorted by diff desc
        p              (float):       percentage threshold for induction heads (default=3%)
        tick_step      (int):         step for x‐axis ticks (default=2)
    """
    # --- Induction heads (TOP-p%) globally flattened ---
    num_layers = len(avg_scores)
    num_heads  = avg_scores[0].shape[0]

    # flatten all layer/head scores into one 1D tensor
    all_flat = torch.cat([s for s in avg_scores])                      # shape (L*H,)

    # number of heads to pick
    print(num_layers * num_heads * (p / 100))
    n_ablate = math.ceil(num_layers * num_heads * (p / 100))
    print(n_ablate)

    # get indices of top-n_ablate scores in the flattened vector
    topk_vals, topk_idx = torch.topk(all_flat, n_ablate, largest=True)

    # now decode flat indices back to (layer, head)
    layers = topk_idx // num_heads
    heads  = topk_idx % num_heads

    # count how many of those top‐indices fall in each layer
    ind_counts = np.bincount(layers.cpu().numpy(), minlength=num_layers)

    # normalize by total induction heads (== n_ablate)
    ind_ratio = ind_counts / ind_counts.sum()

    # --- Repetition neurons (unchanged) ---
    total_neurons = len(sorted_neurons)
    n_rep         = 1000
    top_rep       = sorted_neurons[:n_rep]
    rep_layers    = [info['neuron'][0] for info in top_rep]
    rep_counts     = np.array([rep_layers.count(l) for l in range(num_layers)])
    rep_ratio      = rep_counts / rep_counts.sum()

    # --- Plot ---
    x = np.arange(num_layers)
    ticks = np.arange(0, num_layers, tick_step)

    plt.figure()
    plt.bar(x, rep_ratio, label=f'Rep. Neurons (K={n_rep})')
    plt.plot(x, ind_ratio, '-o', color='darkorange', label=f'Ind. Heads (K={n_ablate})')
    plt.xlabel("Layer Index", fontsize=12)
    plt.ylabel("Fraction of Top Units", fontsize=12)
    #plt.title(f"Top Repetition Neurons vs. Top Induction Heads by Layer")
    plt.xticks(ticks, fontsize=12)
    plt.yticks(fontsize=12)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(f"/workspace/nhi/figure/report/llama_31_8b/layer_comparison.png",format="png",
       dpi=300,
       bbox_inches="tight")
    plt.show()
plot_repetition_bar_induction_line(sortedHeads, sortedNeurons)

In [ ]:
sortedHeads = induction_heads.compute_prefix_matching_scores(model, tokenizer)

In [ ]:
len(sortedHeads[0])

In [ ]:
from tqdm import tqdm
import importlib
import utils
importlib.reload(induction_heads)
importlib.reload(utils)

def id_ab(model, tokenizer, shot=10, seed=42):
    tasks = ['rep', 'rec', 'ceb', 'wsq3','wsq4']
    d = {}
    sortedHeads = induction_heads.compute_prefix_matching_scores(model, tokenizer)
    for task in tqdm(tasks):
        file_data = f"/workspace/nhi/data/repetition_icl/2_apr/{task}_{shot}shot.jsonl"
        ab_dataset = load_data(file_data)
        ground_truths = [item['ground_truth'] for item in ab_dataset]
        nor_answer = utils.generate_answer(model, tokenizer, ab_dataset, seed)
        ab_hd_result = induction_heads.ablate_attention_heads_8(model, tokenizer, sortedHeads, ab_dataset, seed=seed)
        #induction_heads.ablate_attention_heads_loglikelihood(model, tokenizer, sortedHeads, ab_dataset)
        nor_ =  utils.extract_answers_nor(nor_answer, shots = shot)
        hd = utils.extract_answers_ab(ab_hd_result, shots=shot)
        d[task] = {
            'ground_truths': ground_truths,
            'hd': hd,
            'nor_': nor_,
        }
        #print(seed)
    return d
hd_result = id_ab(model, tokenizer, shot=10, seed = 41)
#hd_result = id_ab(model, tokenizer, shot=10, seed = 42)

In [ ]:
hd_result42 = load_back("/workspace/nhi/data/ab_result/llama31_8b/10_shot/id_only/hd_result_42.jsonl", exp = "hd")

In [ ]:
hd_result42['rep']['hd']['induction'].keys()

In [ ]:
import json

output_path = "/workspace/nhi/data/ab_result/llama31_8b/10_shot/id_only/hd_result_41.jsonl"
with open(output_path, "w", encoding="utf-8") as fout:
    for task, res in hd_result.items():
        # embed the task name in the record
        record = {"task": task, **res}
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
def compute_head_ablation_accuracy(hd_result):
    accuracies = {}
    for task, res in hd_result.items():
        gts     = res['ground_truths']
        nor_p   = res['nor_']
        n       = len(gts)
        # baseline (no ablation) accuracy
        nor_acc = sum(1 for gt, p in zip(gts, nor_p) if gt == p) / n

        hd_acc = {}
        for mode, counts_map in res['hd'].items():
            mode_acc = {}
            for k, preds in counts_map.items():
                # pair up ground truth and preds
                length = min(n, len(preds))
                correct = sum(1 for i in range(length) if gts[i] == preds[i])
                mode_acc[k] = correct / length if length > 0 else float('nan')
            hd_acc[mode] = mode_acc

        accuracies[task] = {
            'nor_acc': nor_acc,
            'hd_acc':  hd_acc
        }

    return accuracies

# Example usage:
#hd_result_extract = id_ab(model, tokenizer)
head_accs = compute_head_ablation_accuracy(hd_result)

In [ ]:
hd_result['rep']['hd']['induction'].keys()

In [ ]:
import matplotlib.pyplot as plt
def plot_head_ablation_results(accs):
    tasks    = list(accs.keys())
    k_values = sorted(accs[tasks[0]]['hd_acc']['induction'].keys())
    xs       = [0] + k_values   # prepend 0 for the baseline

    fig, axes = plt.subplots(1, len(tasks), sharey=True, figsize=(len(tasks)*2, 3))
    for ax, task in zip(axes, tasks):
        data = accs[task]
        nor  = data['nor_acc']
        ind  = [data['hd_acc']['induction'][k] for k in k_values]
        rnd  = [data['hd_acc']['random']   [k] for k in k_values]

        # prepend baseline
        ind_plot = [nor] + ind
        rnd_plot = [nor] + rnd
        ax.set_box_aspect(1)
        ax.plot(xs, ind_plot, 'o-',  label='Induction', markersize=4)
        ax.plot(xs, rnd_plot, 's--', label='Random',    markersize=4)

        ax.set_title(task, fontsize=9)
        ax.set_xticks(xs)
        ax.set_xlabel("Percentage of Heads Ablated", fontsize=8)
        ax.grid(axis='y', linestyle='--', alpha=0.5)


    axes[0].set_ylabel("Accuracy", fontsize=9)

    # shared legend
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc='lower center',
        ncol=2, frameon=True,
        fontsize=8, title="Ablation Type", title_fontsize=9,
        bbox_to_anchor=(0.5, -0.08)
    )

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.savefig("/workspace/nhi/figure/natural/llama_3.1_8b/10_shot/41/ab_head.png",format="png",
           dpi=300,
           bbox_inches="tight")
    plt.show()
rename_map = {
    'wsq3': 'wsq1',
    'wsq4': 'wsq2',
}

# 2) build a new dict with keys replaced
head_acc_renamed = {
    rename_map.get(task, task): results
    for task, results in head_accs.items()
}
plot_head_ablation_results(head_acc_renamed)

In [ ]:
def compute_head_ablation_accuracy_by_class(hd_result):
    out = {}
    for task, res in hd_result.items():
        gts   = res['ground_truths']
        nor_p = res['nor_']
        n = len(gts)
        # baseline per-class
        nor = {}
        for cls in ('Foo','Bar'):
            idxs = [i for i,gt in enumerate(gts) if gt==cls]
            if idxs:
                nor[cls] = sum(1 for i in idxs if nor_p[i]==cls) / len(idxs)
            else:
                nor[cls] = float('nan')

        hd_acc = {}
        for mode, cmap in res['hd'].items():
            mode_map = {}
            for k, preds in cmap.items():
                m = {}
                for cls in ('Foo','Bar'):
                    # pick positions where gt==cls, but also within preds length
                    idxs = [i for i,gt in enumerate(gts) if gt==cls and i < len(preds)]
                    if idxs:
                        m[cls] = sum(1 for i in idxs if preds[i]==cls) / len(idxs)
                    else:
                        m[cls] = float('nan')
                mode_map[k] = m
            hd_acc[mode] = mode_map

        out[task] = {
            'nor':    nor,
            'hd_acc': hd_acc
        }
    return out

ab_hd_acc = compute_head_ablation_accuracy_by_class(hd_result)

In [ ]:
def plot_head_ablation_by_class(accs):
    tasks    = list(accs.keys())
    k_vals   = sorted(accs[tasks[0]]['hd_acc']['induction'].keys())
    xs       = [0] + k_vals

    # map your internal classes to pretty titles
    class_titles = {'Foo': "Pattern", 'Bar': "Non-Pattern"}

    fig, axes = plt.subplots(2, len(tasks), sharey=True, figsize=(len(tasks)*2, 4))
    for col, task in enumerate(tasks):
        data = accs[task]
        nor  = data['nor']       # {'Foo':…, 'Bar':…}

        for row, cls in enumerate(['Foo','Bar']):
            ax = axes[row, col]
            # build the two curves
            ind_curve = [nor[cls]] + [data['hd_acc']['induction'][k][cls] for k in k_vals]
            rnd_curve = [nor[cls]] + [data['hd_acc']['random'][k][cls]    for k in k_vals]
            
            ax.plot(xs, ind_curve, 'o-', label='Induction Heads', markersize=4)
            ax.plot(xs, rnd_curve, 's--', label='Random Heads',    markersize=4)
            
            # use the pretty class title
            if col == 0:
                ax.set_ylabel(class_titles[cls] + " Accuracy", fontsize=9)
            if row == 0:
                ax.set_title(task, fontsize=9)
            #ax.set_title(task, fontsize=9)
            ax.set_xticks(xs)
            ax.grid(axis='y', linestyle='--', alpha=0.5)

    # shared legend
    handles, labels = axes[0,-1].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        ncol=2,
        loc='upper center',
        bbox_to_anchor=(0.5, 0.1),
        frameon=True, fontsize=8,
        title="Ablation Type", title_fontsize=9
    )#loc='lower center',

    fig.supxlabel("Number of Heads Ablated", fontsize=10 , y = 0.1)
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.savefig("/workspace/nhi/figure/natural/llama_3.1_8b/10_shot/41/ab_head_class.png",format="png",
            dpi=300,
            bbox_inches="tight")
    

    plt.show()
rename_map = {
    'wsq3': 'wsq1',
    'wsq4': 'wsq2',
}

# 2) build a new dict with keys replaced
ab_hd_acc_renamed = {
    rename_map.get(task, task): results
    for task, results in ab_hd_acc.items()
}
# Call it:
plot_head_ablation_by_class(ab_hd_acc_renamed)

In [ ]:
def plot_ablation_drops(head_by_class, rep_results,
                        rep_mode='top', rep_K=250, head_k=3):
    """
    head_by_class: output of compute_head_ablation_accuracy_by_class()
    rep_results:   your repetition-neuron results dict
    rep_mode:      'top'
    rep_K:         integer, e.g. 250
    head_k:        integer, e.g. 3
    """
    tasks = list(head_by_class.keys())
    segs  = ['0_0.2', '0.4_0.6', '0.8_1.0']

    # compute repetition‐neuron metrics
    rep_metrics = compute_ablation_accuracies(rep_results,
                                             mode=rep_mode,
                                             K=rep_K)

    # collect drops
    pat_drop_heads = []
    non_drop_heads = []
    pat_drop_rep   = []
    non_drop_rep   = []

    for t in tasks:
        # baseline
        pat_base = head_by_class[t]['nor']['Foo']
        non_base = head_by_class[t]['nor']['Bar']

        # induction heads @ head_k
        pat_h = head_by_class[t]['hd_acc']['induction'][head_k]['Foo']
        non_h = head_by_class[t]['hd_acc']['induction'][head_k]['Bar']
        pat_drop_heads.append(pat_base - pat_h)
        non_drop_heads.append(non_base - non_h)

        # repetition neurons @ rep_K (average across segments)
        pat_vals = [rep_metrics[t][seg]['Foo'] for seg in segs]
        non_vals = [rep_metrics[t][seg]['Bar'] for seg in segs]
        pat_mean = np.mean(pat_vals)
        non_mean = np.mean(non_vals)
        pat_drop_rep.append(pat_base - pat_mean)
        non_drop_rep.append(non_base - non_mean)

    # plot
    x = np.arange(len(tasks))
    width = 0.35
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(8, 5))

    # Pattern drops
    ax1.bar(x - width/2, pat_drop_heads, width, label=f'Heads (k={head_k})')
    ax1.bar(x + width/2, pat_drop_rep,   width, label=f'RepNeurons (K={rep_K})')
    ax1.set_ylabel("Pattern Drop")
    ax1.set_title("Drop in Pattern Accuracy")
    # allow negative drops if any
    all_pat = pat_drop_heads + pat_drop_rep
    ax1.set_ylim(min(all_pat)*1.1, max(all_pat)*1.1)
    ax1.legend(frameon=False)

    # Non-Pattern drops
    ax2.bar(x - width/2, non_drop_heads, width, label=f'Heads (k={head_k})')
    ax2.bar(x + width/2, non_drop_rep,  width, label=f'RepNeurons (K={rep_K})')
    ax2.set_ylabel("Non-Pattern Drop")
    ax2.set_title("Drop in Non-Pattern Accuracy")
    ax2.set_xticks(x)
    ax2.set_xticklabels(tasks)
    # allow negative drops
    all_non = non_drop_heads + non_drop_rep
    ax2.set_ylim(min(all_non)*1.1, max(all_non)*1.1)
    ax2.legend(frameon=False)

    ax2.set_xlabel("Task")
    plt.tight_layout()
    plt.show()

# Usage:
# head_by_class = compute_head_ablation_accuracy_by_class(hd_result)all_task_result
plot_ablation_drops(ab_hd_acc, d)

In [ ]:
import pandas as pd
import numpy as np

def make_ablation_diff_table_last_seg(
    head_by_class,
    rep_results,
    head_k=3,
    rep_mode='top',
    hd_mode='induction',
    rep_K=250,
    last_seg = '0.8_1.0',
    dual_accs_class=None
):
    # reuse your existing compute_ablation_accuracies
    def compute_ablation_accuracies(results_dict, mode='top', K=250):
        def class_accuracy(gt_list, pred_list, cls):
            idxs = [i for i, gt in enumerate(gt_list) if gt == cls]
            if not idxs:
                return float('nan')
            correct = sum(1 for i in idxs if pred_list[i] == cls)
            return correct / len(idxs)

        acc = {}
        for task, res in results_dict.items():
            gts = res['ground_truths']
            segs = res['nr_layer'][mode][K]
            task_acc = {}
            for seg, preds in segs.items():
                task_acc[seg] = {
                    'Foo': class_accuracy(gts, preds, 'Foo'),
                    'Bar': class_accuracy(gts, preds, 'Bar')
                }
            # also store baseline
            task_acc['Foo_nor'] = class_accuracy(gts, res['nor_'], 'Foo')
            task_acc['Bar_nor'] = class_accuracy(gts, res['nor_'], 'Bar')
            acc[task] = task_acc
        return acc

    rep_met = compute_ablation_accuracies(rep_results, mode=rep_mode, K=rep_K)

    

    rows = []
    rename_map = {
    'wsq3': 'wsq1',
    'wsq4': 'wsq2',
    }

    rep_met = {
    rename_map.get(task, task): results
    for task, results in rep_met.items()
    }
    head_by_class = {
        rename_map.get(task, task): results
        for task, results in head_by_class.items()
    }
    if dual_accs_class:
            top_deltas = extract_last_segment_delta(dual_accs_class,
                                            rep_mode='top',
                                            mask_mode=hd_mode,
                                            neuron_count=rep_K,
                                            pct=head_k,
                                            segment="0.8->1")
            # random_deltas = extract_last_segment_delta(dual_accs_class,
            #                                 rep_mode='random',
            #                                 mask_mode=hd_mode,
            #                                 neuron_count=rep_K,
            #                                 pct=head_k,
            #                                 segment=last_seg)
    for task, info in head_by_class.items():
        # baseline
        pat0 = info['nor']['Foo']
        non0 = info['nor']['Bar']
        # heads after-intervention
        pat_h = info['hd_acc'][hd_mode][head_k]['Foo']
        non_h = info['hd_acc'][hd_mode][head_k]['Bar']
        # repetition neurons after-intervention (last segment only)
        pat_r = rep_met[task][last_seg]['Foo']
        non_r = rep_met[task][last_seg]['Bar']

        #ab_dual

        if dual_accs_class:
            top_pat_ir = top_deltas[task][0]
            top_non_pat_ir = top_deltas[task][1]

            tmp = {
            'Task': task,
            # after - before
            f'Δ Pattern (H={head_k})': round(pat_h - pat0, 3),
            f'Δ Pattern (R={rep_K})': round(pat_r - pat0, 3),
            f'Δ Pattern (HR={head_k}_{rep_K})': round(top_pat_ir, 3),
            f'Δ Pattern (HR={head_k}_{rep_K})': round(top_pat_ir, 3),
            f'Δ Non-Pattern (H={head_k})': round(non_h - non0, 3),
            f'Δ Non-Pattern (R={rep_K})': round(non_r - non0, 3),
            f'Δ Non-Pattern (HR={head_k}_{rep_K})': round(top_non_pat_ir, 3),
        }
        else:
            tmp = {
            'Task': task,
            # after - before
            f'Δ Pattern (H={head_k})': round(pat_h - pat0, 3),
            f'Δ Pattern (R={rep_K})': round(pat_r - pat0, 3),
            f'Δ Non-Pattern (H={head_k})': round(non_h - non0, 3),
            f'Δ Non-Pattern (R={rep_K})': round(non_r - non0, 3),
        }

        rows.append(tmp)
    df = pd.DataFrame(rows).set_index('Task')
    return df

# Usage:
df_lastseg = make_ablation_diff_table_last_seg(ab_hd_acc, d, head_k=1, rep_mode='top',
                                               hd_mode='induction', last_seg = '0.8_1.0',
                                               dual_accs_class=dual_accs_class, rep_K=250)
print(df_lastseg.to_string())


In [ ]:
import pandas as pd
import numpy as np

def make_ablation_diff_table_last_seg(
    head_by_class,
    rep_results,
    head_k=3,
    rep_mode='top',
    hd_mode='induction',
    rep_K=250,
    last_seg = '0.8_1.0',
    dual_accs_class=None
):
    # reuse your existing compute_ablation_accuracies
    def compute_ablation_accuracies(results_dict, mode='top', K=250):
        def class_accuracy(gt_list, pred_list, cls):
            idxs = [i for i, gt in enumerate(gt_list) if gt == cls]
            if not idxs:
                return float('nan')
            correct = sum(1 for i in idxs if pred_list[i] == cls)
            return correct / len(idxs)

        acc = {}
        for task, res in results_dict.items():
            gts = res['ground_truths']
            segs = res['nr_layer'][mode][K]
            task_acc = {}
            for seg, preds in segs.items():
                task_acc[seg] = {
                    'Foo': class_accuracy(gts, preds, 'Foo'),
                    'Bar': class_accuracy(gts, preds, 'Bar')
                }
            # also store baseline
            task_acc['Foo_nor'] = class_accuracy(gts, res['nor_'], 'Foo')
            task_acc['Bar_nor'] = class_accuracy(gts, res['nor_'], 'Bar')
            acc[task] = task_acc
        return acc

    rep_met = compute_ablation_accuracies(rep_results, mode=rep_mode, K=rep_K)

    

    rows = []
    rename_map = {
    'wsq3': 'wsq1',
    'wsq4': 'wsq2',
    }

    rep_met = {
    rename_map.get(task, task): results
    for task, results in rep_met.items()
    }
    head_by_class = {
        rename_map.get(task, task): results
        for task, results in head_by_class.items()
    }
    if dual_accs_class:
            top_deltas = extract_last_segment_delta(dual_accs_class,
                                            rep_mode=rep_mode,
                                            mask_mode=hd_mode,
                                            neuron_count=rep_K,
                                            pct=head_k,
                                            segment="0.8->1") #"0.8->1"0.4->0.6
            # random_deltas = extract_last_segment_delta(dual_accs_class,
            #                                 rep_mode='random',
            #                                 mask_mode=hd_mode,
            #                                 neuron_count=rep_K,
            #                                 pct=head_k,
            #                                 segment=last_seg)
    for task, info in head_by_class.items():
        # baseline
        pat0 = info['nor']['Foo']
        non0 = info['nor']['Bar']
        # heads after-intervention
        pat_h = info['hd_acc'][hd_mode][head_k]['Foo']
        non_h = info['hd_acc'][hd_mode][head_k]['Bar']
        # repetition neurons after-intervention (last segment only)
        pat_r = rep_met[task][last_seg]['Foo']
        non_r = rep_met[task][last_seg]['Bar']

        #ab_dual

        if dual_accs_class:
            top_pat_ir = top_deltas[task][0]
            top_non_pat_ir = top_deltas[task][1]
            # rad_pat_ir = random_deltas[task][0]
            # rad_non_pat_ir = top_deltas[task][1]
            tmp = {
            'Task': task,
            # after - before
            f'Δ Pattern (H={head_k})': round(pat_h - pat0, 3),
            f'Δ Pattern (R={rep_K})': round(pat_r - pat0, 3),
            f'Δ Pattern (HR={head_k}_{rep_K})': round(top_pat_ir, 3),
            f'Δ Pattern (HR={head_k}_{rep_K})': round(top_pat_ir, 3),
            f'Δ Non-Pattern (H={head_k})': round(non_h - non0, 3),
            f'Δ Non-Pattern (R={rep_K})': round(non_r - non0, 3),
            f'Δ Non-Pattern (HR={head_k}_{rep_K})': round(top_non_pat_ir, 3),
        }
        else:
            tmp = {
            'Task': task,
            # after - before
            f'Δ Pattern (H={head_k})': round(pat_h - pat0, 3),
            f'Δ Pattern (R={rep_K})': round(pat_r - pat0, 3),
            f'Δ Non-Pattern (H={head_k})': round(non_h - non0, 3),
            f'Δ Non-Pattern (R={rep_K})': round(non_r - non0, 3),
        }

        rows.append(tmp)
    df = pd.DataFrame(rows).set_index('Task')
    return df

# Usage:
df_lastseg = make_ablation_diff_table_last_seg(ab_hd_acc, d, head_k=3, rep_mode='top',
                                               hd_mode='induction', last_seg = '0.8_1.0',
                                               dual_accs_class=dual_accs_class, rep_K=250)
print(df_lastseg.to_string())


In [ ]:
file_paths = {
        "nor_answer": "nor_answer.jsonl",
        "ab_nr_all_result": "ab_nr_all_result.jsonl",
        "ab_nr_layer_result": "ab_nr_layer_result.jsonl",
        "ab_hd_result": "ab_hd_result.jsonl"
    }

task_name = "ceb" # rep, rec, ceb, wsq3, wsq4
#model_name = 'Qwen/Qwen2.5-7B'  # Change this to 'meta-llama/Llama-3.1-8B' to test the other case
shots = 10
utils.save_experiment_results(nor_answer, ab_nr_all_result, ab_nr_layer_result, ab_hd_result, file_paths, task_name, model_name, shots)

In [ ]:
nor_answer, ab_nr_all_result,ab_hd_result

## Joint Ablation

In [ ]:
from tqdm import tqdm
import importlib
import utils
importlib.reload(repetition_neurons)
importlib.reload(utils)

def joint_ab(model, tokenizer, sortedNeurons, shot=10, seed=42):
    tasks = ['rep', 'rec', 'ceb', 'wsq3','wsq4']
    d = {}
    sortedHeads = induction_heads.compute_prefix_matching_scores(model, tokenizer)
    for task in tqdm(tasks):
        file_data = f"/workspace/nhi/data/repetition_icl/2_apr/{task}_{shot}shot.jsonl"
        ab_dataset = load_data(file_data)#random.sample(load_data(file_data),100)#[:100]
        ground_truths = [item['ground_truth'] for item in ab_dataset]
        nor_answer = utils.generate_answer(model, tokenizer, ab_dataset, seed)
        
        dual_results = utils.run_dual_ablation_study(
            model=model,
            tokenizer=tokenizer,
            dataset=ab_dataset,          
            sortedNeurons=sortedNeurons,
            id_avg_scores=sortedHeads,   
            percent_list=[1, 3, 5],
            segment_ranges=[(0, 0.2), (0.4, 0.6), (0.8, 1.0)],
            neurons_to_ablate_list=[100, 250],
            shot=shot,
        )
        nor_ =  utils.extract_answers_nor(nor_answer, shots = shot)
        #ab_dual = utils.extract_answers_ab_layer(ab_dual_result, shots=10)
        d[task] = {
            'ground_truths': ground_truths,
            'ab_dual':      dual_results,
            'nor_':          nor_,
        }
    return d

joint_rs = joint_ab(model, tokenizer, sortedNeurons, shot=10, seed=41)


In [ ]:
joint_rs

In [ ]:
joint_rs['rep']['ab_dual']['top']['induction'].keys()

In [ ]:
import json

output_path = "/workspace/nhi/data/ab_result/llama31_8b/10_shot/dual_ab/ab_joint_results_41.jsonl"
with open(output_path, "w", encoding="utf-8") as fout:
    for task, res in joint_rs.items():
        # embed the task name in the record
        record = {"task": task, **res}
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
joint_rs = load_back("/workspace/nhi/data/ab_result/llama31_8b/10_shot/dual_ab/ab_joint_results_42.jsonl", exp='ab_dual')

In [ ]:
joint_rs['rep']['ab_dual']['top']['induction'][100].keys()

In [ ]:
def compute_dual_ablation_accuracies(joint_results):
    accuracies = {}
    for task, info in joint_results.items():
        gts = info['ground_truths']
        n   = len(gts)

        # 1) baseline accuracy
        nor_preds = info['nor_']
        baseline = sum(1 for gt, p in zip(gts, nor_preds) if gt == p) / n

        # 2) collect joint‐ablation accuracies
        ab_acc = {}
        for rep_mode, rep_dict in info['ab_dual'].items():
            ab_acc[rep_mode] = {}
            for mask_mode, mask_dict in rep_dict.items():
                ab_acc[rep_mode][mask_mode] = {}
                for k, pct_dict in mask_dict.items():
                    ab_acc[rep_mode][mask_mode][k] = {}
                    for pct, seg_results in pct_dict.items():
                        seg_acc = {}
                        for seg_res in seg_results:
                            seg   = seg_res['segment']
                            preds = seg_res['results']
                            # accuracy over the entire dataset
                            acc = sum(1 for gt, p in zip(gts, preds) if gt == p) / n
                            seg_acc[seg] = acc
                        ab_acc[rep_mode][mask_mode][k][pct] = seg_acc

        accuracies[task] = {
            'baseline_acc': baseline,
            'ablation_acc': ab_acc
        }

    return accuracies

In [ ]:
dual_accs = compute_dual_ablation_accuracies(joint_rs)
print(dual_accs['rep']['baseline_acc'])                              # baseline
print(dual_accs['rep']['ablation_acc']['top']['induction'][100][3])

In [ ]:
def compute_dual_ablation_accuracies_by_class(joint_results):
    """
    From joint_results (with keys 'ground_truths', 'nor_', 'ab_dual'),
    compute for each task:
      - baseline Foo / Bar accuracy
      - for each rep_mode ∈ {'top','random'}, each mask_mode ∈ {'induction','random'},
        each k ∈ neurons_to_ablate_list, each pct ∈ percent_list:
          segment → { 'Foo': acc, 'Bar': acc }
    """
    out = {}
    for task, info in joint_results.items():
        gts  = info['ground_truths']
        n    = len(gts)
        # count examples per class
        foo_idxs = [i for i,gt in enumerate(gts) if gt=='Foo']
        bar_idxs = [i for i,gt in enumerate(gts) if gt=='Bar']

        # baseline
        nor = info['nor_']
        base_foo = sum(1 for i in foo_idxs if nor[i]=='Foo')/len(foo_idxs) if foo_idxs else float('nan')
        base_bar = sum(1 for i in bar_idxs if nor[i]=='Bar')/len(bar_idxs) if bar_idxs else float('nan')

        ab = {}
        for rep_mode, rep_block in info['ab_dual'].items():
            ab[rep_mode] = {}
            for mask_mode, mask_block in rep_block.items():
                ab[rep_mode][mask_mode] = {}
                for k, pct_block in mask_block.items():
                    ab[rep_mode][mask_mode][k] = {}
                    for pct, seg_list in pct_block.items():
                        seg_acc = {}
                        for seg_res in seg_list:
                            seg   = seg_res['segment']
                            preds = seg_res['results']
                            # compute Foo / Bar acc on this segment
                            foo_acc = sum(1 for i in foo_idxs if preds[i]=='Foo')/len(foo_idxs) if foo_idxs else float('nan')
                            bar_acc = sum(1 for i in bar_idxs if preds[i]=='Bar')/len(bar_idxs) if bar_idxs else float('nan')
                            #if foo_acc is n
                            #for i 
                            seg_acc[seg] = {'Foo': foo_acc, 'Bar': bar_acc}
                        ab[rep_mode][mask_mode][k][pct] = seg_acc

        out[task] = {
            'baseline': {'Foo': base_foo, 'Bar': base_bar},
            'ablation': ab
        }

    return out

# Example usage:
dual_accs_class = compute_dual_ablation_accuracies_by_class(joint_rs)
# Now dual_accs['rep']['baseline'] → {'Foo':…, 'Bar':…}
# and dual_accs['rep']['ablation']['top']['induction'][100][1]['0->0.2'] → {'Foo':…, 'Bar':…}


In [ ]:
def debug_labels(info, segment=None):
    gts = info['ground_truths']
    if segment:                # e.g. segment = (0.4, 0.6)
        start, stop = segment
        sl = slice(int(start*len(gts)), int(stop*len(gts)))
        gts = gts[sl]

    from collections import Counter
    print(task)
    print(Counter(gts))

# whole‑task label counts
#debug_labels(joint_rs['wsq3'])          # → probably {'Foo': N, 'Bar': 0}

# same for a failing segment:
#debug_labels(joint_rs['wsq3'], (0.8,1)) # → likely {'Foo': N}

for task in joint_rs:
    debug_labels(joint_rs[task])


In [ ]:
def js_acc_all(joint_rs):
    """
    Build a dictionary mapping
      (task, rep_mode, mask_mode, k, pct, segment) -> accuracy

    Parameters
    ----------
    joint_rs : dict
        Nested results object of the form shown earlier.

    Returns
    -------
    dict
        acc_dict with tuple keys and float accuracies.
    """
    acc_dict = {}

    for task, info in joint_rs.items():
        gts = info['ground_truths']                       # list of ground‑truth labels
        for rep_mode, rep_dict in info['ab_dual'].items():
            for mask_mode, mask_dict in rep_dict.items():
                for k, pct_dict in mask_dict.items():
                    k = int(k)                            # ensure numeric
                    for pct, seg_results in pct_dict.items():
                        pct = int(pct)
                        for seg_res in seg_results:
                            segment  = seg_res['segment'] # e.g. "0->0.2"
                            preds    = seg_res['results']

                            accuracy = sum(gt == p for gt, p in zip(gts, preds)) / len(gts)
                            key = (task, rep_mode, mask_mode, k, pct, segment)
                            acc_dict[key] = accuracy

    return acc_dict

z = js_acc_all(joint_rs)

In [ ]:
def extract_last_segment_delta(dual_accs,
                               rep_mode='top',
                               mask_mode='induction',
                               neuron_count=250,
                               pct=3,
                               segment='0.8->1',
                               ndigits=3):
    """
    Returns a dict task → {'Foo': ΔFoo, 'Bar': ΔBar}, 
    where Δ = Acc_after(segment) - Acc_baseline,
    rounded to ndigits decimals.
    """
    out = {}
    for task, info in dual_accs.items():
        base = info['baseline']            # {'Foo':…, 'Bar':…}
        after = ( info['ablation']
                      [rep_mode]
                      [mask_mode]
                      [neuron_count]
                      [pct]
                      [segment] )
        out[task] = (
            round(after['Foo'] - base['Foo'], ndigits),#'Foo': 
            round(after['Bar'] - base['Bar'], ndigits) #'Bar': 
        )
    rename_map = {
    'wsq3': 'wsq1',
    'wsq4': 'wsq2',
}

    # 2) build a new dict with keys replaced
    out = {
        rename_map.get(task, task): results
        for task, results in out.items()
    }
    return out

# Usage:
deltas = extract_last_segment_delta(dual_accs_class,
                                    rep_mode='top',
                                    mask_mode='induction',
                                    neuron_count=250,
                                    pct=1,
                                    segment='0.8->1')
# rename_map = {
#     'wsq3': 'wsq1',
#     'wsq4': 'wsq2',
# }

# # 2) build a new dict with keys replaced
# deltas = {
#     rename_map.get(task, task): results
#     for task, results in deltas.items()
# }

In [ ]:
deltas['rep'][0]

In [ ]:
def mean_delta_over_segments(dual_accs,
                             rep_mode: str,
                             mask_mode: str,
                             neuron_count: int,
                             pct: int,
                             ndigits: int = 3):
    """
    For each task in dual_accs, compute
      mean( acc_after(seg) ) – acc_baseline
    over the segments ['0->0.2','0.4->0.6','0.8->1'], separately for Foo and Bar.
    """
    segments = ['0->0.2', '0.4->0.6', '0.8->1']
    out = {}
    for task, info in dual_accs.items():
        base = info['baseline']  # {'Foo':…, 'Bar':…}
        foo_sum = 0.0
        bar_sum = 0.0
        for seg in segments:
            after = ( info['ablation']
                          [rep_mode]
                          [mask_mode]
                          [neuron_count]
                          [pct]
                          [seg] )
            foo_sum += after['Foo']
            bar_sum += after['Bar']
        foo_avg = foo_sum / len(segments)
        bar_avg = bar_sum / len(segments)
        out[task] = (
            round(foo_avg - base['Foo'], ndigits),#'Foo': 
            round(bar_avg - base['Bar'], ndigits) #'Bar': 
        )
    return out

# Example usage:
mean_deltas = mean_delta_over_segments(
    dual_accs=dual_accs_class,
    rep_mode='random',
    mask_mode='induction',
    neuron_count=250,
    pct=1
)
rename_map = {
    'wsq3': 'wsq1',
    'wsq4': 'wsq2',
}

# 2) build a new dict with keys replaced
mean_deltas = {
    rename_map.get(task, task): results
    for task, results in mean_deltas.items()
}
print(mean_deltas)
# e.g. {'rep': {'Foo': 0.689, 'Bar': 0.653}, ...}


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

tasks = ['rep','rec','ceb','wsq1','wsq2']
top_deltas = extract_last_segment_delta(dual_accs_class,
                                    rep_mode='top',
                                    mask_mode='random',
                                    neuron_count=250,
                                    pct=3,
                                    segment='0.8->1')
random_deltas = extract_last_segment_delta(dual_accs_class,
                                    rep_mode='random',
                                    mask_mode='random',
                                    neuron_count=250,
                                    pct=3,
                                    segment='0.8->1')
# unpack
pattern_top  = [top_deltas[t][0] for t in tasks]
pattern_rand = [random_deltas[t][0] for t in tasks]
non_top      = [top_deltas[t][1] for t in tasks]
non_rand     = [random_deltas[t][1] for t in tasks]

# styling for one‐column width (≈3.3in)
plt.rcParams.update({
    'figure.figsize': (3.3, 3.0),
    'axes.labelsize': 8,
    'xtick.labelsize': 9,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'lines.linewidth': 0.9,
})

x = np.arange(len(tasks))
width = 0.35

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

# Pattern‐accuracy Δ
ax1.bar(x - width/2, pattern_top,  width, label='Top',    alpha=0.8)
ax1.bar(x + width/2, pattern_rand, width, label='Random', alpha=0.8)
ax1.set_ylabel('Δ Pattern')
#ax1.legend(ncol=2, loc='upper right', frameon=False)

# Non‐pattern Δ
ax2.bar(x - width/2, non_top,  width, label='Repetition',    alpha=0.8)
ax2.bar(x + width/2, non_rand, width, label='Random', alpha=0.8)
ax2.set_ylabel('Δ Non-Pattern')
ax2.set_xticks(x)
ax2.set_xticklabels(tasks, fontsize=10, rotation=0)
ax2.legend(ncol=2, loc='upper right', frameon=True, fontsize=6,
    framealpha=0.5,  title="Ablation Neuron", title_fontsize=6)

fig.supxlabel('Task', fontsize=9)
plt.tight_layout(pad=0.5)
plt.savefig("/workspace/nhi/figure/natural/llama_3.1_8b/10_shot/43/dual_250_3_rndhd.png",format="png",
            dpi=300,
            bbox_inches="tight")

plt.show()


In [ ]:
for task, info in dual_accs_class.items():
        base = info['baseline']
        print(task, base)

# Visualize

In [ ]:
import importlib

importlib.reload(utils)
importlib.reload(visualize)

deactivated_neurons = utils.extract_deactivated_neurons(ab_nr_layer_result['top'])
diffs_by_group = utils.analyze_diffs_of_deactivated_neurons(deactivated_neurons, sortedNeurons)
visualize.plot_diffs_distribution_by_group_2(diffs_by_group, '/workspace/nhi/figure/ab_layer/llama_31_8b/ceb')

In [ ]:
df_2 = utils.analyze_exp_2(ab_dataset, nor_, nr_layer)
overview_results = utils.analyze_class_overview(df_2)
overview_results_top, overview_results_random = utils.split_results_by_mode(overview_results)

In [ ]:
import importlib

importlib.reload(visualize)
visualize.plot_accuracy_for_two_labels(df_2, '/workspace/nhi/figure/ab_layer/llama_31_8b/ceb')

In [ ]:
len(nr_layer['top'][20]['0_0.2'])

# Draft

Extract 250 pattern and non-pattern sample for wsq

In [ ]:
import json

INPUT_PATH  = "/workspace/nhi/data/repetition_icl/25_feb/wordseq31_5shot_test_cases.jsonl"           # <- your input JSONL
OUTPUT_PATH = "/workspace/nhi/data/repetition_icl/2_apr/wsq3_5shot.jsonl"   # <- where to save the 500 examples

foo_items = []
bar_items = []

with open(INPUT_PATH, "r", encoding="utf-8") as fin:
    for line in fin:
        obj = json.loads(line)
        gt = obj.get("ground_truth", "")
        if   gt == "Foo" and len(foo_items) < 250:
            foo_items.append(obj)
        elif gt == "Bar" and len(bar_items) < 250:
            bar_items.append(obj)
        # stop early once we have both sets
        if len(foo_items) == 250 and len(bar_items) == 250:
            break

# combine and write out
selected = foo_items + bar_items
with open(OUTPUT_PATH, "w", encoding="utf-8") as fout:
    for obj in selected:
        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

print(f"Written {len(foo_items)} Foo and {len(bar_items)} Bar examples to {OUTPUT_PATH}")

In [ ]:
import torch
import torch.nn.functional as F

def score_loglikelihood(model, tokenizer, prompt: str, continuation: str, device=None) -> float:
    """
    Tính tổng log‐likelihood của continuation khi nối vào prompt,
    bằng cách dùng labels để chỉ tính loss trên phần continuation.
    Trả về giá trị log‐likelihood (dương nếu continuation hợp lý).
    """
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    # 1) Mã hoá prompt và continuation
    input_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
    cont_ids  = tokenizer(continuation, return_tensors="pt", add_special_tokens=False).input_ids.to(device)

    # 2) Ghép lại
    ids = torch.cat([input_ids, cont_ids], dim=1)            # [1, L_ctx + L_cont]
    seq_len = ids.size(1)
    L_ctx   = input_ids.size(1)
    L_cont  = cont_ids.size(1)

    # 3) Chuẩn bị labels: chỉ tính loss từ vị trí L_ctx trở đi
    labels = ids.clone()
    labels[:, :L_ctx] = -100   # -100 là ignore_index cho cross‐entropy

    with torch.no_grad():
        outputs = model(ids, labels=labels)
        # outputs.loss = trung bình -log p trên L_cont token
        avg_neglog = outputs.loss.item()  
    # 4) Tổng log‐likelihood = - avg_neglog * số token
    return - avg_neglog * L_cont


def predict_mc(model, tokenizer, prompts: list[str], choices: list[str], device=None) -> list[int]:
    """
    Với mỗi prompt, tính log‐likelihood cho mọi choice, chọn index có score cao nhất.
    Trả về list các index dự đoán.
    """
    preds = []
    for prompt in prompts:
        scores = [score_loglikelihood(model, tokenizer, prompt, c, device=device) 
                  for c in choices]
        # chọn index của choice có score cao nhất
        pred = int(torch.tensor(scores).argmax())
        preds.append(pred)
    return preds

In [ ]:
prompt = (
        "Below is a list of various sequences. Your task is to classify each sequence.\n\n"
        "Y K Y K: Foo\n"
        "I Y P X: Bar\n"
        "X K X K: Foo\n"
        "N L N L: Foo\n"
        "R S R S: Foo\n"
        "U Z U Z:"
    )
choices = [" Foo", " Bar"]  

ll_foo = score_loglikelihood(model, tokenizer, prompt, choices[0])
ll_bar = score_loglikelihood(model, tokenizer, prompt, choices[1])
print(f"log-lik(Foo) = {ll_foo:.3f}, log-lik(Bar) = {ll_bar:.3f}")

pred = predict_mc(model, tokenizer, [prompt], choices)[0]
print("Predicted:", choices[pred])

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns

def show_o_proj_change(model, layer_idx, heads):
    """
    Visualise o_proj.weight before/after ablation for the specified heads.
    Automatically restores the weights even if an error happens.
    """
    if isinstance(heads, int):
        heads = [heads]

    layer  = model.model.layers[layer_idx].self_attn
    W_full = layer.o_proj.weight.detach().cpu()          # [d_out, H*Dh]
    Dh     = layer.head_dim

    def extract_cols(t, head_list):
        return torch.cat([t[:, h*Dh:(h+1)*Dh] for h in head_list], dim=1)

    W_before = extract_cols(W_full, heads)

    originals = None          # define here so it’s visible in finally
    try:
        # -------- ablate
        originals = induction_heads.disable_Wo_heads(model, {layer_idx: heads})
        W_after   = extract_cols(layer.o_proj.weight.detach().cpu(), heads)
        W_diff    = (W_before - W_after).abs()

        # -------- plot
        fig, ax = plt.subplots(1, 3, figsize=(14, 4))
        for i, (m, title) in enumerate(zip(
                [W_before, W_after, W_diff],
                ["Before ablation", "After ablation", "Absolute diff"])):
            sns.heatmap(m.abs().numpy(), ax=ax[i], cmap="viridis", cbar=False)
            ax[i].set_title(title)
            ax[i].set_xlabel("Dim inside Wo_h")
            ax[i].set_ylabel("Output dim")
        plt.tight_layout()
        plt.show()

    finally:
        # -------- always restore
        if originals is not None:
            induction_heads.restore_Wo_heads(model, originals)

# -----------------------------------------------------------
# example usage: heads 0 and 3 in layer 10
show_o_proj_change(model, layer_idx=10, heads=[0, 3])


In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------------
# Ablation helpers (identical to the ones you already wrote)
# ---------------------------------------------------------------------
def disable_Wo_heads(model, block_config):
    """
    Zero‑out each Wo_h (o_proj columns) specified in block_config.

    block_config = {layer_idx: [head_idx, head_idx, ...], ...}
    Returns a dict so we can restore later.
    """
    original = {}
    for layer_idx, heads in block_config.items():
        layer = model.model.layers[layer_idx].self_attn
        Dh    = layer.head_dim
        W     = layer.o_proj.weight                        # [d_out, H*Dh]
        for h in heads:
            s, e = h * Dh, (h + 1) * Dh
            original[(layer_idx, h)] = W[:, s:e].data.clone()
            W[:, s:e].data.zero_()
    return original


def restore_Wo_heads(model, original):
    """Undo `disable_Wo_heads` using the returned dict."""
    for (layer_idx, h), block in original.items():
        layer = model.model.layers[layer_idx].self_attn
        Dh    = layer.head_dim
        s, e  = h * Dh, (h + 1) * Dh
        layer.o_proj.weight.data[:, s:e] = block


# ---------------------------------------------------------------------
# Visualisation
# ---------------------------------------------------------------------
def show_o_proj_change(model, layer_idx, heads, zoom_only=False):
    """
    Visualise the effect of ablating heads in o_proj.weight.

    • heads can be int or list[int]
    • If zoom_only=True, skip the whole‑matrix plot.
    """
    if isinstance(heads, int):
        heads = [heads]

    layer  = model.model.layers[layer_idx].self_attn
    W_full = layer.o_proj.weight.detach().cpu()            # [d_out, H*Dh]
    Dh     = layer.head_dim

    def extract_cols(t, head_list):
        """Concatenate the column blocks for the given heads."""
        return torch.cat([t[:, h * Dh : (h + 1) * Dh] for h in head_list], dim=1)

    # snapshot before
    W_before = extract_cols(W_full, heads)

    originals = None
    try:
        # ----- ablate -----
        originals = disable_Wo_heads(model, {layer_idx: heads})
        W_after   = extract_cols(layer.o_proj.weight.detach().cpu(), heads)
        W_diff    = (W_before - W_after).abs()

        # shared colour scale for the zoom plots
        vmin, vmax = W_before.abs().min(), W_before.abs().max()

        # ========== PLOT ==========
        ncols = 3 + (0 if zoom_only else 1)
        fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 5))

        col = 0
        if not zoom_only:
            ax = axes[col]
            sns.heatmap(W_full.abs().numpy(), ax=ax, cmap="viridis", cbar=False)
            ax.set_title(f"Whole W_o (layer {layer_idx})")
            ax.set_xlabel("Head * d_h")
            ax.set_ylabel("Output dim")
            # outline ablated blocks in red
            for h in heads:
                s, e = h * Dh, (h + 1) * Dh
                ax.add_patch(
                    plt.Rectangle(
                        (s, 0), e - s, W_full.shape[0],
                        fill=False, edgecolor="red", lw=1.5
                    )
                )
            col += 1

        for m, title in zip(
            [W_before, W_after, W_diff],
            ["Before ablation (slice)", "After ablation (slice)", "|Before‑After|"]
        ):
            sns.heatmap(
                m.abs().numpy(), ax=axes[col],
                cmap="viridis", vmin=vmin, vmax=vmax, cbar=False
            )
            axes[col].set_title(title)
            axes[col].set_xlabel("Dim inside Wo_h")
            axes[col].set_ylabel("Output dim")
            col += 1

        plt.tight_layout()
        plt.show()

    finally:
        if originals is not None:
            restore_Wo_heads(model, originals)


# ---------------------------------------------------------------------
# Example usage
# ---------------------------------------------------------------------
# Visualise heads 0 & 3 in layer 10; show full matrix + zoom
show_o_proj_change(model, layer_idx=30, heads=[10, 3])

# If you only want the zoom‑in triptych, set zoom_only=True
# show_o_proj_change(model, layer_idx=10, heads=[0, 3], zoom_only=True)


In [ ]:
# def wrap_attn_forward(original_forward, layer_idx, model, mask_layer):
#     """
#     Wraps the attention forward function to inject a custom attention mask.
#     Ablates entire heads where mask_layer is 0, using vectorized operations.

#     Args:
#         original_forward: The original forward function of the attention layer.
#         layer_idx (int): Index of the current layer.
#         model: The transformer model.
#         mask_layer (torch.Tensor): Mask indicating which heads to ablate.

#     Returns:
#         Callable: Wrapped forward function with the injected attention mask.
#     """
#     @functools.wraps(original_forward)
#     def wrapped_forward(**kwargs):
#         hs = kwargs['hidden_states'] 
#         seq_len = hs.shape[1]
#         #print('Error here')

#         attn_mask = base_causal_mask[:, :, :seq_len, :seq_len].clone()

#         ablate_mask = (mask_layer == 0.0) 
#         if ablate_mask.any():
#             attn_mask[:, ablate_mask, :, :] = 0


#         if isinstance(model.config.torch_dtype, str):
#             dtype_str = model.config.torch_dtype.lower()
#             if dtype_str in ["float16", "fp16"]:
#                 torch_dtype = torch.float16
#             elif dtype_str in ["float32", "fp32"]:
#                 torch_dtype = torch.float32
#             elif dtype_str in ["bfloat16", "bf16"]:
#                 torch_dtype = torch.bfloat16
#             else:
#                 torch_dtype = torch.float32
#         else:
#             torch_dtype = model.config.torch_dtype

#         attn_mask = (1.0 - attn_mask) * torch.finfo(torch_dtype).min
#         attn_mask = attn_mask.to(hs.device)
        

#         kwargs["attention_mask"] = attn_mask

#         return original_forward(**kwargs)
#     return wrapped_forward

In [ ]:
model.config

In [ ]:
layer

In [ ]:
layer = model.model.layers[0].self_attn
print(model.config.num_attention_heads)      # Should be 32
print(layer.head_dim)       # Should be 128
print(layer.o_proj.weight.shape)  # Should be [4096, 4096]

In [ ]:
prompt = "The quick brown fox jumps over the lazy dog. The quick brown fox jumps"
def generate_output(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

# Baseline output
baseline = generate_output(model, tokenizer, prompt)
print("Baseline Output:", baseline)
def get_logits(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.logits  # Shape: [batch_size, seq_len, vocab_size]

#
logits_baseline = get_logits(model, tokenizer, prompt)

block_config = {10: [4]}
orig_weights = disable_Wo_heads(model, block_config)

logits_ablated = get_logits(model, tokenizer, prompt)


restore_Wo_heads(model, orig_weights)

# Compare logits (e.g., for the last token)
last_token_logits_diff = (logits_baseline[0, -1] - logits_ablated[0, -1]).abs().mean()
print("Mean logit difference:", last_token_logits_diff.item())